In [1]:
# ===============================
# Cell 1: Imports, Device, Reproducibility
# ===============================

import os
import random
import warnings
from glob import glob
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm import tqdm

# Image processing
import cv2
from PIL import Image

# Scientific / QUS utilities
from scipy.spatial.distance import cdist
from scipy.spatial import ConvexHull
from scipy import ndimage

from skimage import segmentation as skimage_seg
from skimage.measure import label, regionprops
from skimage.feature import graycomatrix, graycoprops

# Torch & torchvision
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as T
import torchvision.models as models

# Machine learning utilities for QUS-SVM teacher
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
    classification_report
)

# Saving/loading SVM/scaler if needed
import joblib

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Reproducibility
warnings.filterwarnings("ignore")


def seed_everything(seed=42):
    """
    Make training more reproducible.
    Some CUDA operations may still be slightly nondeterministic.
    """
    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    os.environ["PYTHONHASHSEED"] = str(seed)

    torch.backends.cudnn.deterministic = False
    torch.backends.cudnn.benchmark = True


SEED = 42
seed_everything(SEED)

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Using device: cuda
GPU: Tesla T4


In [ ]:
# ===============================
# Cell 2: True QUS Feature Estimator
# ===============================
# 8 QUS features:
# ED, SAF, TH, ASR, CVX, SOLIDITY, CCONTRAST, LBD
# ===============================

import numpy as np
import cv2

from scipy.spatial import ConvexHull, cKDTree
from skimage import segmentation as skimage_seg
from skimage.measure import label, regionprops


QUS_FEATURE_NAMES = [
    "ED",
    "SAF",
    "TH",
    "ASR",
    "CVX",
    "SOLIDITY",
    "CCONTRAST",
    "LBD"
]


def ensure_binary_mask(mask):
    """
    Convert mask to binary 0/1 uint8.
    Works for masks stored as 0/255, 0/1, or float masks.
    """
    if mask is None:
        raise ValueError("Mask is None.")

    if mask.ndim == 3:
        mask = cv2.cvtColor(mask, cv2.COLOR_BGR2GRAY)

    if mask.max() > 1:
        mask = (mask > 127).astype(np.uint8)
    else:
        mask = (mask > 0).astype(np.uint8)

    return mask


def ensure_grayscale_image(image):
    """
    Convert image to grayscale float32.
    """
    if image is None:
        raise ValueError("Image is None.")

    if image.ndim == 3:
        image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    return image.astype(np.float32)


def keep_largest_component(mask):
    """
    Keep only the largest connected lesion component.
    This protects QUS features from small noisy mask islands.
    """
    mask = ensure_binary_mask(mask)

    labeled = label(mask)
    props = regionprops(labeled)

    if len(props) == 0:
        return mask

    largest = max(props, key=lambda p: p.area)
    clean_mask = (labeled == largest.label).astype(np.uint8)

    return clean_mask


def sort_coordinates(list_of_xy_coords):
    """
    Sort boundary coordinates approximately clockwise.

    Input:
        [N, 2] array, usually x,y points.
    """
    list_of_xy_coords = np.asarray(list_of_xy_coords)

    if len(list_of_xy_coords) == 0:
        return list_of_xy_coords

    cx, cy = list_of_xy_coords.mean(axis=0)
    x, y = list_of_xy_coords.T

    angles = np.arctan2(y - cy, x - cx)
    indices = np.argsort(angles)

    return list_of_xy_coords[indices]


def get_largest_contour(mask):
    """
    Return largest contour from binary mask.
    OpenCV contour point ordering is x,y.
    """
    mask = ensure_binary_mask(mask)

    contours, _ = cv2.findContours(
        mask,
        cv2.RETR_EXTERNAL,
        cv2.CHAIN_APPROX_NONE
    )

    if len(contours) == 0:
        return None

    return max(contours, key=cv2.contourArea)


def calculate_edge_diffusivity(image, mask, c=20, num_normals=100):
    """
    Edge Diffusivity, ED.

    Uses intensity/gradient behavior across sampled boundary normals.

    Note:
        OpenCV contour point = x,y
        NumPy image indexing = image[y,x]
    """
    image = ensure_grayscale_image(image)
    mask = keep_largest_component(mask)

    contour = get_largest_contour(mask)
    if contour is None:
        return 0.0

    boundary_points = contour[:, 0, :]  # x,y

    if len(boundary_points) < 5:
        return 0.0

    sampled_points = boundary_points[
        np.linspace(
            0,
            len(boundary_points) - 1,
            min(num_normals, len(boundary_points)),
            dtype=int
        )
    ]

    H, W = image.shape[:2]

    gradients_contour = []
    gradients_inner = []
    intensities_contour = []
    intensities_inner = []

    for i in range(len(sampled_points)):
        x, y = sampled_points[i]

        prev_point = sampled_points[i - 1]
        next_point = sampled_points[(i + 1) % len(sampled_points)]

        dx = next_point[0] - prev_point[0]
        dy = next_point[1] - prev_point[1]

        norm = np.sqrt(dx**2 + dy**2)
        if norm <= 1e-8:
            continue

        nx, ny = -dy / norm, dx / norm

        contour_band = []
        inner_band = []

        for t in range(-c // 2, c // 2):
            px = int(round(x + t * nx))
            py = int(round(y + t * ny))

            if 0 <= px < W and 0 <= py < H:
                if t < 0:
                    contour_band.append(image[py, px])
                else:
                    inner_band.append(image[py, px])

        if len(contour_band) > 1 and len(inner_band) > 1:
            gradient_contour = np.mean(np.gradient(contour_band))
            gradient_inner = np.mean(np.gradient(inner_band))

            gradients_contour.append(gradient_contour)
            gradients_inner.append(gradient_inner)
            intensities_contour.append(np.mean(contour_band))
            intensities_inner.append(np.mean(inner_band))

    if len(gradients_contour) == 0 or len(gradients_inner) == 0:
        return 0.0

    avg_gradient_diff = np.mean(
        np.abs(np.array(gradients_contour) - np.array(gradients_inner))
    )

    avg_intensity = np.mean(
        np.array(intensities_contour) + np.array(intensities_inner)
    )

    if avg_intensity <= 1e-8:
        return 0.0

    ed = 1.0 - (avg_gradient_diff / avg_intensity)

    if not np.isfinite(ed):
        return 0.0

    return float(ed)


def calculate_saf(mask):
    """
    Shape Asymmetry Factor, SAF.

    Measures boundary deviation from fitted ellipse.
    """
    mask = keep_largest_component(mask)

    boundaries = skimage_seg.find_boundaries(mask, mode="inner").astype(np.uint8)
    coords = np.where(boundaries == 1)
    coords_yx = np.array(list(zip(coords[0], coords[1])))  # y,x

    if len(coords_yx) < 5:
        return 0.0

    # Convert y,x to x,y for OpenCV.
    boundary_xy = np.stack(
        [coords_yx[:, 1], coords_yx[:, 0]],
        axis=1
    ).astype(np.float32)

    boundary_xy = sort_coordinates(boundary_xy)

    if len(boundary_xy) < 5:
        return 0.0

    try:
        ellipse = cv2.fitEllipse(boundary_xy)
    except cv2.error:
        return 0.0

    x0, y0 = ellipse[0]
    a, b = ellipse[1]
    theta = np.deg2rad(ellipse[2])

    a = a / 2.0
    b = b / 2.0

    if a <= 1e-8 or b <= 1e-8:
        return 0.0

    N = len(boundary_xy) + 150
    t = np.linspace(0, 2 * np.pi, N)

    x = x0 + a * np.cos(theta) * np.cos(t) - b * np.sin(theta) * np.sin(t)
    y = y0 + a * np.sin(theta) * np.cos(t) + b * np.cos(theta) * np.sin(t)

    ellipse_points = np.stack([x, y], axis=1).astype(np.float32)

    if len(ellipse_points) < 5:
        return 0.0

    try:
        tree = cKDTree(ellipse_points)
        distances, _ = tree.query(boundary_xy, k=1)
        distance_sum = np.sum(distances)
    except Exception:
        return 0.0

    area = cv2.contourArea(boundary_xy)

    if area <= 1e-8:
        return 0.0

    saf = distance_sum / area

    if not np.isfinite(saf):
        return 0.0

    return float(saf)


def calculate_heterogeneity(image, mask):
    """
    Tissue Heterogeneity, TH.

    Standard deviation of nonzero B-mode pixels inside lesion.
    """
    image = ensure_grayscale_image(image)
    mask = keep_largest_component(mask)

    tumor_pixels = image[mask == 1]
    nonzero_pixels = tumor_pixels[tumor_pixels != 0]

    if nonzero_pixels.size == 0:
        return 0.0

    th = np.std(nonzero_pixels)

    if not np.isfinite(th):
        return 0.0

    return float(th)


def calculate_aspect_ratio(mask):
    """
    Aspect Ratio, ASR.

    height / width of lesion bounding box.
    """
    mask = keep_largest_component(mask)

    coords = np.argwhere(mask == 1)

    if coords.size == 0:
        return 0.0

    height = coords[:, 0].max() - coords[:, 0].min() + 1
    width = coords[:, 1].max() - coords[:, 1].min() + 1

    if width <= 0:
        return 0.0

    asr = height / width

    if not np.isfinite(asr):
        return 0.0

    return float(asr)


def get_convexity(mask):
    """
    Convexity, CVX.

    CVX = convex hull perimeter / actual lesion perimeter.
    """
    mask = keep_largest_component(mask)

    boundaries = skimage_seg.find_boundaries(mask, mode="inner").astype(np.uint8)
    coords = np.where(boundaries == 1)
    coords_yx = np.array(list(zip(coords[0], coords[1])))  # y,x

    if len(coords_yx) < 5:
        return 0.0

    boundary_xy = np.stack(
        [coords_yx[:, 1], coords_yx[:, 0]],
        axis=1
    ).astype(np.float32)

    boundary_xy = sort_coordinates(boundary_xy)

    try:
        convex_hull = cv2.convexHull(boundary_xy, clockwise=True)
        convex_perimeter = cv2.arcLength(convex_hull, True)
        mask_perimeter = cv2.arcLength(boundary_xy, True)
    except cv2.error:
        return 0.0

    if mask_perimeter <= 1e-8:
        return 0.0

    convexity = convex_perimeter / mask_perimeter

    if not np.isfinite(convexity):
        return 0.0

    return float(convexity)


def calculate_solidity(mask):
    """
    Solidity.

    Solidity = lesion area / convex area.
    """
    mask = keep_largest_component(mask)

    labeled_mask = label(mask)
    props = regionprops(labeled_mask)

    if len(props) == 0:
        return 0.0

    prop = max(props, key=lambda p: p.area)

    lesion_area = float(prop.area)
    convex_area = float(prop.convex_area)

    if convex_area <= 1e-8:
        return 0.0

    solidity = lesion_area / convex_area

    if not np.isfinite(solidity):
        return 0.0

    return float(solidity)


def calculate_cocontrast(image, mask, levels=32):
    """
    Co-occurrence Contrast, CCONTRAST.

    Masked GLCM-style contrast.
    Only pixel pairs where both pixels are inside the lesion are counted.

    This avoids background-zero pixels dominating the GLCM.
    """
    image = ensure_grayscale_image(image)
    mask = keep_largest_component(mask)

    if np.sum(mask) < 10:
        return 0.0

    image = np.clip(image, 0, 255)

    # Quantize image to 0 ... levels-1
    q = (image / 256.0 * levels).astype(np.uint8)
    q = np.clip(q, 0, levels - 1)

    # Directions: 0, 45, 90, 135 degrees
    displacements = [
        (0, 1),
        (1, 1),
        (1, 0),
        (1, -1)
    ]

    contrasts = []

    H, W = q.shape

    for dy, dx in displacements:
        if dx >= 0:
            x_src_start, x_src_end = 0, W - dx
            x_dst_start, x_dst_end = dx, W
        else:
            x_src_start, x_src_end = -dx, W
            x_dst_start, x_dst_end = 0, W + dx

        y_src_start, y_src_end = 0, H - dy
        y_dst_start, y_dst_end = dy, H

        if y_src_end <= y_src_start or x_src_end <= x_src_start:
            continue

        src_mask = mask[y_src_start:y_src_end, x_src_start:x_src_end]
        dst_mask = mask[y_dst_start:y_dst_end, x_dst_start:x_dst_end]

        valid = (src_mask == 1) & (dst_mask == 1)

        if np.sum(valid) == 0:
            continue

        src_vals = q[y_src_start:y_src_end, x_src_start:x_src_end][valid]
        dst_vals = q[y_dst_start:y_dst_end, x_dst_start:x_dst_end][valid]

        contrast = np.mean((src_vals.astype(np.float32) - dst_vals.astype(np.float32)) ** 2)
        contrasts.append(contrast)

    if len(contrasts) == 0:
        return 0.0

    cocontrast = float(np.mean(contrasts))

    if not np.isfinite(cocontrast):
        return 0.0

    return cocontrast


def calculate_lbd(image, mask, band_width=3):
    """
    Lesion Boundary Descriptor, LBD.

    Absolute difference between average outside-boundary-band intensity
    and inside-boundary-band intensity.
    """
    image = ensure_grayscale_image(image)
    mask = keep_largest_component(mask)

    kernel = np.ones((2 * band_width + 1, 2 * band_width + 1), np.uint8)

    dilated = cv2.dilate(mask, kernel, iterations=1)
    eroded = cv2.erode(mask, kernel, iterations=1)

    outer_band = (dilated == 1) & (mask == 0)
    inner_band = (mask == 1) & (eroded == 0)

    if np.sum(outer_band) == 0 or np.sum(inner_band) == 0:
        return 0.0

    outside_mean = np.mean(image[outer_band])
    inside_mean = np.mean(image[inner_band])

    lbd = abs(outside_mean - inside_mean)

    if not np.isfinite(lbd):
        return 0.0

    return float(lbd)


def quantitative_ultrasound_features(image, mask):
    """
    Final paper-matched 8-feature QUS extractor.

    Returns:
        [ED, SAF, TH, ASR, CVX, SOLIDITY, CCONTRAST, LBD]
    """
    image = ensure_grayscale_image(image)
    mask = keep_largest_component(mask)

    ed = calculate_edge_diffusivity(image, mask)
    saf = calculate_saf(mask)
    th = calculate_heterogeneity(image, mask)
    asr = calculate_aspect_ratio(mask)
    cvx = get_convexity(mask)
    solidity = calculate_solidity(mask)
    cocontrast = calculate_cocontrast(image, mask)
    lbd = calculate_lbd(image, mask)

    features = np.array(
        [ed, saf, th, asr, cvx, solidity, cocontrast, lbd],
        dtype=np.float32
    )

    # Safety: replace NaN/Inf with 0
    features = np.nan_to_num(features, nan=0.0, posinf=0.0, neginf=0.0)

    return features


def quantitative_ultrasound_features_dict(image, mask):
    """
    Same as quantitative_ultrasound_features(), but returns dictionary.
    Useful while creating CSV.
    """
    features = quantitative_ultrasound_features(image, mask)

    return {
        name: float(value)
        for name, value in zip(QUS_FEATURE_NAMES, features)
    }

In [ ]:
# ===============================
# Cell 3: Multi-CAM Loss
# ===============================
# Components:
# Neighbor BCE loss + foreground IoU loss + background IoU loss + boundary loss
# Works for B-mode CAM, H-scan CAM, Nakagami CAM
# ===============================

import torch
import torch.nn as nn
import torch.nn.functional as F


def minmax_normalize_cam(cam, eps=1e-6):
    """
    Per-sample min-max normalization for CAM maps.

    Input:
        cam: [B, 1, H, W]

    Output:
        cam normalized to [0,1] per sample.
    """
    B = cam.shape[0]
    flat = cam.reshape(B, -1)

    cam_min = flat.min(dim=1)[0].reshape(B, 1, 1, 1)
    cam_max = flat.max(dim=1)[0].reshape(B, 1, 1, 1)

    cam = (cam - cam_min) / (cam_max - cam_min + eps)
    return cam


def prepare_cam_and_mask(
    cam,
    mask,
    apply_sigmoid=False,
    normalize_cam=False,
    eps=1e-6
):
    """
    Prepare CAM and mask for CAM-based loss.

    cam:
        Tensor shape [B, 1, Hc, Wc] or [B, Hc, Wc]
        If raw logits are passed, set apply_sigmoid=True.
        If CAM is not nicely scaled, set normalize_cam=True.

    mask:
        Tensor shape [B, 1, H, W] or [B, H, W]
        Binary mask, values can be 0/1 or 0/255.

    Returns:
        cam  -> [B, 1, H, W], float, clamped
        mask -> [B, 1, H, W], float, binary
    """

    if cam.dim() == 3:
        cam = cam.unsqueeze(1)

    if mask.dim() == 3:
        mask = mask.unsqueeze(1)

    cam = cam.float()
    mask = mask.float().to(cam.device)

    # Convert 0/255 mask to 0/1 if needed
    if mask.max() > 1:
        mask = mask / 255.0

    mask = (mask > 0.5).float()

    # Resize CAM to mask size
    if cam.shape[-2:] != mask.shape[-2:]:
        cam = F.interpolate(
            cam,
            size=mask.shape[-2:],
            mode="bilinear",
            align_corners=False
        )

    if apply_sigmoid:
        cam = torch.sigmoid(cam)

    if normalize_cam:
        cam = minmax_normalize_cam(cam, eps=eps)

    cam = torch.clamp(cam, eps, 1.0 - eps)

    return cam, mask


def make_boundary_distance_map_from_mask(mask, eps=1e-6):
    """
    Simple torch-based distance-like map from mask.

    Penalizes activation outside lesion more than inside lesion.

    Input:
        mask: [B, 1, H, W], binary 0/1

    Output:
        dist_map: [B, 1, H, W]
    """
    if mask.dim() == 3:
        mask = mask.unsqueeze(1)

    mask = (mask > 0.5).float()

    # Boundary approximation using local variation
    max_pool = F.max_pool2d(mask, kernel_size=3, stride=1, padding=1)
    min_pool = -F.max_pool2d(-mask, kernel_size=3, stride=1, padding=1)
    boundary = (max_pool - min_pool > 0).float()

    # Background penalty map: background gets 1, lesion gets smaller value.
    # Boundary region gets intermediate emphasis.
    bg = 1.0 - mask
    dist_map = bg + 0.5 * boundary

    B = dist_map.shape[0]
    flat = dist_map.reshape(B, -1)
    max_val = flat.max(dim=1)[0].reshape(B, 1, 1, 1)

    dist_map = dist_map / (max_val + eps)

    return dist_map


class BoundaryLoss(nn.Module):
    """
    Boundary loss:
        L_bd = mean(CAM * distance_map)

    dist_map:
        Tensor shape [B, 1, H, W] or [B, H, W]

    If dist_map is None:
        returns 0 by default.
    """

    def __init__(self, auto_from_mask=False):
        super().__init__()
        self.auto_from_mask = auto_from_mask

    def forward(self, cam, mask=None, dist_map=None):
        if dist_map is None:
            if self.auto_from_mask and mask is not None:
                dist_map = make_boundary_distance_map_from_mask(mask)
            else:
                return cam.new_tensor(0.0)

        if dist_map.dim() == 3:
            dist_map = dist_map.unsqueeze(1)

        dist_map = dist_map.float().to(cam.device)

        if dist_map.shape[-2:] != cam.shape[-2:]:
            dist_map = F.interpolate(
                dist_map,
                size=cam.shape[-2:],
                mode="bilinear",
                align_corners=False
            )

        # Normalize per sample to avoid very large loss values
        B = dist_map.shape[0]
        flat = dist_map.reshape(B, -1)
        max_val = flat.max(dim=1)[0].reshape(B, 1, 1, 1)
        dist_map = dist_map / (max_val + 1e-6)

        loss = cam * dist_map
        return loss.mean()


class ForegroundIoULoss(nn.Module):
    """
    Lesion/foreground IoU loss:
        -IoU(CAM, mask)

    Negative Iou.
    Minimizing this encourages IoU to become high.
    """

    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, cam, mask):
        B = cam.shape[0]

        cam_flat = cam.reshape(B, -1)
        mask_flat = mask.reshape(B, -1)

        intersection = (cam_flat * mask_flat).sum(dim=1)
        union = cam_flat.sum(dim=1) + mask_flat.sum(dim=1) - intersection

        iou = (intersection + self.smooth) / (union + self.smooth)

        return (-iou).mean()


class BackgroundIoULoss(nn.Module):
    """
    Background IoU loss:
        -IoU(1-CAM, 1-mask)

    This discourages CAM activation in background.
    """

    def __init__(self, smooth=1.0):
        super().__init__()
        self.smooth = smooth

    def forward(self, cam, mask):
        B = cam.shape[0]

        cam_bg = 1.0 - cam
        mask_bg = 1.0 - mask

        cam_bg_flat = cam_bg.reshape(B, -1)
        mask_bg_flat = mask_bg.reshape(B, -1)

        intersection = (cam_bg_flat * mask_bg_flat).sum(dim=1)
        union = cam_bg_flat.sum(dim=1) + mask_bg_flat.sum(dim=1) - intersection

        iou = (intersection + self.smooth) / (union + self.smooth)

        return (-iou).mean()


class NeighborLoss(nn.Module):
    """
    Stable boundary-aware neighbor BCE loss.

    Boundary pixels receive larger weight.
    Other pixels receive weight 1.
    """

    def __init__(self, boundary_weight=10.0, eps=1e-6):
        super().__init__()
        self.boundary_weight = boundary_weight
        self.eps = eps

    def forward(self, cam, mask):
        """
        cam:  [B, 1, H, W], probability map
        mask: [B, 1, H, W], binary mask
        """

        cam = torch.clamp(cam, self.eps, 1.0 - self.eps)
        mask = mask.float().to(cam.device)

        # Detect local mask variation using maxpool/minpool.
        max_pool = F.max_pool2d(mask, kernel_size=3, stride=1, padding=1)
        min_pool = -F.max_pool2d(-mask, kernel_size=3, stride=1, padding=1)

        boundary_region = (max_pool - min_pool > 0).float()

        weight_map = 1.0 + (self.boundary_weight - 1.0) * boundary_region

        bce = F.binary_cross_entropy(
            cam,
            mask,
            reduction="none"
        )

        weighted_bce = weight_map * bce

        return weighted_bce.mean()


class SingleCAMLoss(nn.Module):
    """
    CAM loss for one modality.

    L_CAM = lambda_neighbor * NeighborLoss
          + lambda_iou_fg  * ForegroundIoU
          + lambda_iou_bg  * BackgroundIoU
          + lambda_boundary * BoundaryLoss

    Recommended first setting:
        lambda_boundary = 0.0

    Later:
        lambda_boundary = 0.05 or 0.1
    """

    def __init__(
        self,
        lambda_neighbor=1.0,
        lambda_iou_fg=1.0,
        lambda_iou_bg=1.0,
        lambda_boundary=0.0,
        boundary_weight=10.0,
        apply_sigmoid=False,
        normalize_cam=False,
        auto_boundary_from_mask=False
    ):
        super().__init__()

        self.lambda_neighbor = lambda_neighbor
        self.lambda_iou_fg = lambda_iou_fg
        self.lambda_iou_bg = lambda_iou_bg
        self.lambda_boundary = lambda_boundary

        self.apply_sigmoid = apply_sigmoid
        self.normalize_cam = normalize_cam

        self.neighbor_loss = NeighborLoss(boundary_weight=boundary_weight)
        self.iou_fg_loss = ForegroundIoULoss()
        self.iou_bg_loss = BackgroundIoULoss()
        self.boundary_loss = BoundaryLoss(auto_from_mask=auto_boundary_from_mask)

    def forward(self, cam, mask, dist_map=None, return_parts=False):
        cam, mask = prepare_cam_and_mask(
            cam,
            mask,
            apply_sigmoid=self.apply_sigmoid,
            normalize_cam=self.normalize_cam
        )

        loss_neighbor = self.neighbor_loss(cam, mask)
        loss_iou_fg = self.iou_fg_loss(cam, mask)
        loss_iou_bg = self.iou_bg_loss(cam, mask)
        loss_boundary = self.boundary_loss(cam, mask=mask, dist_map=dist_map)

        total = (
            self.lambda_neighbor * loss_neighbor
            + self.lambda_iou_fg * loss_iou_fg
            + self.lambda_iou_bg * loss_iou_bg
            + self.lambda_boundary * loss_boundary
        )

        if return_parts:
            return total, {
                "neighbor": loss_neighbor.detach(),
                "iou_fg": loss_iou_fg.detach(),
                "iou_bg": loss_iou_bg.detach(),
                "boundary": loss_boundary.detach(),
                "total": total.detach()
            }

        return total


class MultiCAMLoss(nn.Module):
    """
    Multi-modality CAM loss.

    Expected CAM dictionary:
        cams = {
            "bmode": cam_b,
            "hscan": cam_h,
            "nakagami": cam_n
        }

    mask:
        Same GT lesion mask used for all modalities.

    dist_map:
        Optional distance map.
        Same dist_map can be used for all modalities.
    """

    def __init__(
        self,
        lambda_neighbor=1.0,
        lambda_iou_fg=1.0,
        lambda_iou_bg=1.0,
        lambda_boundary=0.0,
        boundary_weight=10.0,
        apply_sigmoid=False,
        normalize_cam=False,
        auto_boundary_from_mask=False,
        modality_weights=None
    ):
        super().__init__()

        self.single_cam_loss = SingleCAMLoss(
            lambda_neighbor=lambda_neighbor,
            lambda_iou_fg=lambda_iou_fg,
            lambda_iou_bg=lambda_iou_bg,
            lambda_boundary=lambda_boundary,
            boundary_weight=boundary_weight,
            apply_sigmoid=apply_sigmoid,
            normalize_cam=normalize_cam,
            auto_boundary_from_mask=auto_boundary_from_mask
        )

        if modality_weights is None:
            modality_weights = {
                "bmode": 1.0,
                "hscan": 1.0,
                "nakagami": 1.0
            }

        self.modality_weights = modality_weights

    def forward(self, cams, mask, dist_map=None, return_parts=False):
        """
        cams can be either:
            dict: {"bmode": cam_b, "hscan": cam_h, "nakagami": cam_n}

        or:
            list/tuple: [cam_b, cam_h, cam_n]
        """

        if isinstance(cams, (list, tuple)):
            if len(cams) != 3:
                raise ValueError("If cams is list/tuple, expected [cam_b, cam_h, cam_n].")

            cams = {
                "bmode": cams[0],
                "hscan": cams[1],
                "nakagami": cams[2]
            }

        if not isinstance(cams, dict):
            raise TypeError("cams must be a dict or list/tuple.")

        total_loss = None
        parts = {}
        used_modalities = 0

        for name, cam in cams.items():
            if cam is None:
                continue

            weight = float(self.modality_weights.get(name, 1.0))

            loss_m, part_m = self.single_cam_loss(
                cam,
                mask,
                dist_map=dist_map,
                return_parts=True
            )

            weighted_loss_m = weight * loss_m

            if total_loss is None:
                total_loss = weighted_loss_m
            else:
                total_loss = total_loss + weighted_loss_m

            used_modalities += 1
            parts[name] = part_m

        if total_loss is None or used_modalities == 0:
            raise ValueError("No valid CAMs were provided to MultiCAMLoss.")

        # Average over available modalities
        total_loss = total_loss / used_modalities

        if return_parts:
            parts["multicam_total"] = total_loss.detach()
            return total_loss, parts

        return total_loss

In [ ]:
# ===============================
# Cell 4: Squeeze-and-Excitation Block
# ===============================

class SEBlock(nn.Module):
    """
    Channel attention / squeeze-and-excitation block.

    Input:
        x: [B, C, H, W]

    Output:
        channel-reweighted x
    """

    def __init__(self, n_features, reduction=8):
        super(SEBlock, self).__init__()

        hidden = max(n_features // reduction, 1)

        self.avg_pool = nn.AdaptiveAvgPool2d(1)

        self.fc = nn.Sequential(
            nn.Linear(n_features, hidden, bias=True),
            nn.ReLU(inplace=True),
            nn.Linear(hidden, n_features, bias=True),
            nn.Sigmoid()
        )

    def forward(self, x):
        b, c, _, _ = x.shape

        y = self.avg_pool(x).view(b, c)
        y = self.fc(y).view(b, c, 1, 1)

        return x * y



class SpatialAttention(nn.Module):
    """
    Simple spatial attention block.

    Input:
        x: [B, C, H, W]

    Output:
        spatially reweighted x
    """

    def __init__(self, kernel_size=7):
        super().__init__()

        assert kernel_size in [1, 3, 5, 7], "kernel_size should be one of [1, 3, 5, 7]"

        padding = kernel_size // 2

        self.conv = nn.Conv2d(
            in_channels=2,
            out_channels=1,
            kernel_size=kernel_size,
            padding=padding,
            bias=False
        )

        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        max_result, _ = torch.max(x, dim=1, keepdim=True)
        avg_result = torch.mean(x, dim=1, keepdim=True)

        result = torch.cat([max_result, avg_result], dim=1)

        attention = self.sigmoid(self.conv(result))

        return x * attention


def get_resnet18_backbone():
    """
    Loads ResNet18 with compatibility across torchvision versions.
    """
    try:
        weights = models.ResNet18_Weights.DEFAULT
        resnet = models.resnet18(weights=weights)
    except Exception:
        resnet = models.resnet18(pretrained=True)

    return resnet


class prep_model(nn.Module):
    """
    Lightweight preprocessing / input enhancement module.

    Original idea:
        Input image -> shallow ResNet18 feature extraction -> attention -> 1-channel enhancement map
        Then optionally add this enhancement map to the third input channel.

    Input:
        x: [B, 3, H, W]

    Output:
        enhanced single-channel image: [B, 1, H, W]
    """

    def __init__(
        self,
        dropout=False,
        channel_attention=True,
        spatial_attention=True,
        add_sub=True
    ):
        super().__init__()

        self.drop = dropout
        self.channel_attention = channel_attention
        self.spatial_attention = spatial_attention
        self.addsub = add_sub

        resnet = get_resnet18_backbone()

        # Proper ResNet18 early feature extractor:
        # conv1 -> bn1 -> relu -> maxpool -> layer1
        # Output channels after layer1 = 64
        self.resnet_stem_layer1 = nn.Sequential(
            resnet.conv1,
            resnet.bn1,
            resnet.relu,
            resnet.maxpool,
            resnet.layer1
        )

        self.dropout = nn.Dropout2d(p=0.5, inplace=False)

        self.seb = SEBlock(64, reduction=8)
        self.sp = SpatialAttention(kernel_size=7)

        self.conv_out = nn.Conv2d(64, 1, kernel_size=1)

    def forward(self, x):
        """
        x: [B, 3, H, W]
        """

        # Keep original third channel as identity.
        # Your earlier code used x[:, 2:3, :, :], so we keep that behavior.
        identity = x[:, 2:3, :, :]

        _, _, h, w = x.shape

        feat = self.resnet_stem_layer1(x)

        if self.drop:
            feat = self.dropout(feat)

        if self.spatial_attention:
            spa = self.sp(feat)
            feat = feat + spa

        if self.channel_attention:
            feat = self.seb(feat)

        enhancement = self.conv_out(feat)
        enhancement = F.interpolate(
            enhancement,
            size=(h, w),
            mode="bilinear",
            align_corners=False
        )

        if self.addsub:
            enhancement = identity + enhancement

        return enhancement

In [ ]:
# ===============================
# Cell 5: PTL-CNN B-mode Branch
# ===============================

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
from torchvision.models import (
    Inception_V3_Weights,
    ResNet50_Weights,
    VGG16_BN_Weights
)



class PTLCNN_BMODE(nn.Module):
    def __init__(
        self,
        use_prep: bool = True,
        donor_pretrained: bool = True,
        freeze_donors: bool = True,
        inception_pretrained: bool = True,
        dropout_p: float = 0.5,
        norm_mean: float = 0.275,
        norm_std: float = 0.197,
    ):
        super().__init__()

        self.use_prep = use_prep
        self.freeze_donors = freeze_donors

        # ----- DCIEN / prep -----
        if use_prep:
            self.preprocessor = prep_model(
                dropout=False,
                channel_attention=True,
                spatial_attention=True,
                add_sub=True
            )

            self.register_buffer(
                "mean",
                torch.tensor([norm_mean], dtype=torch.float32).view(1, 1, 1, 1)
            )

            self.register_buffer(
                "std",
                torch.tensor([norm_std], dtype=torch.float32).view(1, 1, 1, 1)
            )

        # ----- InceptionV3 -----
        # aux_logits=True is safest when ImageNet weights are used.
        inc_weights = Inception_V3_Weights.IMAGENET1K_V1 if inception_pretrained else None
        self.inception = models.inception_v3(weights=inc_weights, aux_logits=True)

        # ----- Donor backbones -----
        res_weights = ResNet50_Weights.IMAGENET1K_V2 if donor_pretrained else None
        vgg_weights = VGG16_BN_Weights.IMAGENET1K_V1 if donor_pretrained else None

        self.resnet50 = models.resnet50(weights=res_weights)
        self.vgg16bn = models.vgg16_bn(weights=vgg_weights)

        if freeze_donors:
            for p in self.resnet50.parameters():
                p.requires_grad = False

            for p in self.vgg16bn.parameters():
                p.requires_grad = False

            self.resnet50.eval()
            self.vgg16bn.eval()

        # Project donor features to 192 channels to fuse with Inception pre-Mixed_5b.
        # Important: these projection layers are still trainable even when donors are frozen.
        self.proj_res = nn.Conv2d(512, 192, kernel_size=1, bias=False)
        self.proj_vgg = nn.Conv2d(128, 192, kernel_size=1, bias=False)

        self.fuse_bn = nn.BatchNorm2d(192)

        # Final feature refinement + compact feature head
        self.se_final = SEBlock(2048, reduction=8)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))

        self.dropout = nn.Dropout(p=dropout_p, inplace=False)

        self.fc1 = nn.Linear(2048, 512)
        self.fc2 = nn.Linear(512, 128)
        self.fc3 = nn.Linear(128, 64)

        self.act = nn.ReLU(inplace=True)

    def train(self, mode: bool = True):
        """
        Keep frozen donor networks in eval mode even when the full model is set to train().
        """
        super().train(mode)

        if self.freeze_donors:
            self.resnet50.eval()
            self.vgg16bn.eval()

        return self

    def _resnet50_feats(self, x: torch.Tensor):
        """
        ResNet50 donor feature up to layer2.
        Output: [B, 512, H/8, W/8] roughly.
        """
        r = self.resnet50.conv1(x)
        r = self.resnet50.bn1(r)
        r = self.resnet50.relu(r)
        r = self.resnet50.maxpool(r)

        r = self.resnet50.layer1(r)  # 256 ch
        r = self.resnet50.layer2(r)  # 512 ch

        return r

    def _vgg16_feats(self, x: torch.Tensor):
        """
        VGG16-BN donor early feature.
        Output: [B, 128, H/4, W/4] roughly.
        """
        v = self.vgg16bn.features[:14](x)
        return v

    def _hybrid_feats(self, x: torch.Tensor):
        """
        Donor features + trainable 1x1 projections.

        If donors are frozen:
            donor backbone forward is no_grad,
            but projection convs remain trainable.
        """
        if self.freeze_donors:
            with torch.no_grad():
                r = self._resnet50_feats(x)
                v = self._vgg16_feats(x)
        else:
            r = self._resnet50_feats(x)
            v = self._vgg16_feats(x)

        # Projection layers should remain outside no_grad.
        r = self.proj_res(r)  # -> 192 ch
        v = self.proj_vgg(v)  # -> 192 ch

        return r, v

    def _inception_stem_to_pre5b(self, x: torch.Tensor):
        """
        InceptionV3 stem up to pre-Mixed_5b.
        Output: [B, 192, 35, 35] for 299x299 input.
        """
        x = self.inception.Conv2d_1a_3x3(x)
        x = self.inception.Conv2d_2a_3x3(x)
        x = self.inception.Conv2d_2b_3x3(x)
        x = self.inception.maxpool1(x)

        x = self.inception.Conv2d_3b_1x1(x)
        x = self.inception.Conv2d_4a_3x3(x)
        x = self.inception.maxpool2(x)

        return x

    def _inception_tail_from_5b(self, x: torch.Tensor):
        """
        InceptionV3 tail from Mixed_5b to Mixed_7c.
        Output: [B, 2048, 8, 8] for 299x299 input.
        """
        x = self.inception.Mixed_5b(x)
        x = self.inception.Mixed_5c(x)
        x = self.inception.Mixed_5d(x)

        x = self.inception.Mixed_6a(x)
        x = self.inception.Mixed_6b(x)
        x = self.inception.Mixed_6c(x)
        x = self.inception.Mixed_6d(x)
        x = self.inception.Mixed_6e(x)

        x = self.inception.Mixed_7a(x)
        x = self.inception.Mixed_7b(x)
        x = self.inception.Mixed_7c(x)

        return x

    def forward(self, x: torch.Tensor):
        """
        Input:
            x: [B, 3, 299, 299]

        Returns:
            emb128   : [B, 128]   -> used for F-QUS head later
            emb64    : [B, 64]    -> used for fusion
            feat_map : [B, 2048, 8, 8] -> used for CAM head
        """

        # ----- Optional DCIEN preprocessing -----
        if self.use_prep:
            prep = self.preprocessor(x)
            prep = (prep - self.mean) / (self.std + 1e-8)

            # Replace third channel with enhanced channel, keeping first two channels.
            x_in = torch.cat([x[:, 0:2, :, :], prep], dim=1)
        else:
            x_in = x

        # ----- Inception stem -----
        inc_pre5b = self._inception_stem_to_pre5b(x_in)

        # ----- Donor features -----
        r, v = self._hybrid_feats(x_in)

        # ----- Resize donor features and fuse -----
        r = F.interpolate(
            r,
            size=inc_pre5b.shape[-2:],
            mode="bilinear",
            align_corners=False
        )

        v = F.interpolate(
            v,
            size=inc_pre5b.shape[-2:],
            mode="bilinear",
            align_corners=False
        )

        fused = self.fuse_bn(inc_pre5b + r + v)

        # ----- Inception tail -----
        feat_map = self._inception_tail_from_5b(fused)  # [B, 2048, 8, 8]
        feat_map = self.se_final(feat_map)

        # ----- Compact embedding head -----
        z = self.avgpool(feat_map).flatten(1)  # [B, 2048]
        z = self.dropout(z)

        z = self.act(self.fc1(z))       # [B, 512]
        emb128 = self.act(self.fc2(z))  # [B, 128]
        emb64 = self.fc3(emb128)        # [B, 64]

        return emb128, emb64, feat_map

In [ ]:
# ===============================
# Cell 6: PTL-CNN H-scan Branch
# ===============================

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
from torchvision.models import (
    Inception_V3_Weights,
    ResNet50_Weights,
    VGG16_BN_Weights
)



class PTLCNN_HSCAN(nn.Module):
    def __init__(
        self,
        use_prep: bool = True,
        donor_pretrained: bool = True,
        freeze_donors: bool = True,
        inception_pretrained: bool = True,
        dropout_p: float = 0.5,
        norm_mean: float = 0.275,
        norm_std: float = 0.197,
    ):
        super().__init__()

        self.use_prep = use_prep
        self.freeze_donors = freeze_donors

        # ----- DCIEN / prep -----
        if use_prep:
            self.preprocessor = prep_model(
                dropout=False,
                channel_attention=True,
                spatial_attention=True,
                add_sub=True
            )

            self.register_buffer(
                "mean",
                torch.tensor([norm_mean], dtype=torch.float32).view(1, 1, 1, 1)
            )

            self.register_buffer(
                "std",
                torch.tensor([norm_std], dtype=torch.float32).view(1, 1, 1, 1)
            )

        # ----- InceptionV3 -----
        inc_weights = Inception_V3_Weights.IMAGENET1K_V1 if inception_pretrained else None
        self.inception = models.inception_v3(weights=inc_weights, aux_logits=True)

        # ----- Donor backbones -----
        res_weights = ResNet50_Weights.IMAGENET1K_V2 if donor_pretrained else None
        vgg_weights = VGG16_BN_Weights.IMAGENET1K_V1 if donor_pretrained else None

        self.resnet50 = models.resnet50(weights=res_weights)
        self.vgg16bn = models.vgg16_bn(weights=vgg_weights)

        if freeze_donors:
            for p in self.resnet50.parameters():
                p.requires_grad = False

            for p in self.vgg16bn.parameters():
                p.requires_grad = False

            self.resnet50.eval()
            self.vgg16bn.eval()

        # Project donor features to 192 channels to fuse with Inception pre-Mixed_5b.
        # Projection layers remain trainable even when donors are frozen.
        self.proj_res = nn.Conv2d(512, 192, kernel_size=1, bias=False)
        self.proj_vgg = nn.Conv2d(128, 192, kernel_size=1, bias=False)

        self.fuse_bn = nn.BatchNorm2d(192)

        # Final feature refinement + compact feature head
        self.se_final = SEBlock(2048, reduction=8)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))

        self.dropout = nn.Dropout(p=dropout_p, inplace=False)

        self.fc1 = nn.Linear(2048, 512)
        self.fc2 = nn.Linear(512, 128)
        self.fc3 = nn.Linear(128, 64)

        self.act = nn.ReLU(inplace=True)

    def train(self, mode: bool = True):
        """
        Keep frozen donor networks in eval mode even when the full model is set to train().
        """
        super().train(mode)

        if self.freeze_donors:
            self.resnet50.eval()
            self.vgg16bn.eval()

        return self

    def _resnet50_feats(self, x: torch.Tensor):
        """
        ResNet50 donor feature up to layer2.
        """
        r = self.resnet50.conv1(x)
        r = self.resnet50.bn1(r)
        r = self.resnet50.relu(r)
        r = self.resnet50.maxpool(r)

        r = self.resnet50.layer1(r)
        r = self.resnet50.layer2(r)

        return r

    def _vgg16_feats(self, x: torch.Tensor):
        """
        VGG16-BN donor early feature.
        """
        v = self.vgg16bn.features[:14](x)
        return v

    def _hybrid_feats(self, x: torch.Tensor):
        """
        Donor features + trainable 1x1 projections.
        """
        if self.freeze_donors:
            with torch.no_grad():
                r = self._resnet50_feats(x)
                v = self._vgg16_feats(x)
        else:
            r = self._resnet50_feats(x)
            v = self._vgg16_feats(x)

        # Projection layers remain trainable.
        r = self.proj_res(r)
        v = self.proj_vgg(v)

        return r, v

    def _inception_stem_to_pre5b(self, x: torch.Tensor):
        """
        InceptionV3 stem up to pre-Mixed_5b.
        """
        x = self.inception.Conv2d_1a_3x3(x)
        x = self.inception.Conv2d_2a_3x3(x)
        x = self.inception.Conv2d_2b_3x3(x)
        x = self.inception.maxpool1(x)

        x = self.inception.Conv2d_3b_1x1(x)
        x = self.inception.Conv2d_4a_3x3(x)
        x = self.inception.maxpool2(x)

        return x

    def _inception_tail_from_5b(self, x: torch.Tensor):
        """
        InceptionV3 tail from Mixed_5b to Mixed_7c.
        """
        x = self.inception.Mixed_5b(x)
        x = self.inception.Mixed_5c(x)
        x = self.inception.Mixed_5d(x)

        x = self.inception.Mixed_6a(x)
        x = self.inception.Mixed_6b(x)
        x = self.inception.Mixed_6c(x)
        x = self.inception.Mixed_6d(x)
        x = self.inception.Mixed_6e(x)

        x = self.inception.Mixed_7a(x)
        x = self.inception.Mixed_7b(x)
        x = self.inception.Mixed_7c(x)

        return x

    def forward(self, x: torch.Tensor):
        """
        Input:
            x: [B, 3, 299, 299]

        Returns:
            emb128   : [B, 128]
            emb64    : [B, 64]
            feat_map : [B, 2048, 8, 8] -> used for H-scan CAM head
        """

        # ----- Optional DCIEN preprocessing -----
        if self.use_prep:
            prep = self.preprocessor(x)
            prep = (prep - self.mean) / (self.std + 1e-8)

            x_in = torch.cat([x[:, 0:2, :, :], prep], dim=1)
        else:
            x_in = x

        # ----- Inception stem -----
        inc_pre5b = self._inception_stem_to_pre5b(x_in)

        # ----- Donor features -----
        r, v = self._hybrid_feats(x_in)

        # ----- Resize donor features and fuse -----
        r = F.interpolate(
            r,
            size=inc_pre5b.shape[-2:],
            mode="bilinear",
            align_corners=False
        )

        v = F.interpolate(
            v,
            size=inc_pre5b.shape[-2:],
            mode="bilinear",
            align_corners=False
        )

        fused = self.fuse_bn(inc_pre5b + r + v)

        # ----- Inception tail -----
        feat_map = self._inception_tail_from_5b(fused)
        feat_map = self.se_final(feat_map)

        # ----- Compact embedding head -----
        z = self.avgpool(feat_map).flatten(1)
        z = self.dropout(z)

        z = self.act(self.fc1(z))
        emb128 = self.act(self.fc2(z))
        emb64 = self.fc3(emb128)

        return emb128, emb64, feat_map

In [ ]:
# ===============================
# Cell 7: PTL-CNN Nakagami Branch
# ===============================

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
from torchvision.models import (
    Inception_V3_Weights,
    ResNet50_Weights,
    VGG16_BN_Weights
)



class PTLCNN_NAKAGAMI(nn.Module):
    def __init__(
        self,
        use_prep: bool = True,
        donor_pretrained: bool = True,
        freeze_donors: bool = True,
        inception_pretrained: bool = True,
        dropout_p: float = 0.5,
        norm_mean: float = 0.275,
        norm_std: float = 0.197,
    ):
        super().__init__()

        self.use_prep = use_prep
        self.freeze_donors = freeze_donors

        # ----- DCIEN / prep -----
        if use_prep:
            self.preprocessor = prep_model(
                dropout=False,
                channel_attention=True,
                spatial_attention=True,
                add_sub=True
            )

            self.register_buffer(
                "mean",
                torch.tensor([norm_mean], dtype=torch.float32).view(1, 1, 1, 1)
            )

            self.register_buffer(
                "std",
                torch.tensor([norm_std], dtype=torch.float32).view(1, 1, 1, 1)
            )

        # ----- InceptionV3 -----
        inc_weights = Inception_V3_Weights.IMAGENET1K_V1 if inception_pretrained else None
        self.inception = models.inception_v3(weights=inc_weights, aux_logits=True)

        # ----- Donor backbones -----
        res_weights = ResNet50_Weights.IMAGENET1K_V2 if donor_pretrained else None
        vgg_weights = VGG16_BN_Weights.IMAGENET1K_V1 if donor_pretrained else None

        self.resnet50 = models.resnet50(weights=res_weights)
        self.vgg16bn = models.vgg16_bn(weights=vgg_weights)

        if freeze_donors:
            for p in self.resnet50.parameters():
                p.requires_grad = False

            for p in self.vgg16bn.parameters():
                p.requires_grad = False

            self.resnet50.eval()
            self.vgg16bn.eval()

        # Project donor features to 192 channels to fuse with Inception pre-Mixed_5b.
        # Projection layers remain trainable even when donors are frozen.
        self.proj_res = nn.Conv2d(512, 192, kernel_size=1, bias=False)
        self.proj_vgg = nn.Conv2d(128, 192, kernel_size=1, bias=False)

        self.fuse_bn = nn.BatchNorm2d(192)

        # Final feature refinement + compact feature head
        self.se_final = SEBlock(2048, reduction=8)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))

        self.dropout = nn.Dropout(p=dropout_p, inplace=False)

        self.fc1 = nn.Linear(2048, 512)
        self.fc2 = nn.Linear(512, 128)
        self.fc3 = nn.Linear(128, 64)

        self.act = nn.ReLU(inplace=True)

    def train(self, mode: bool = True):
        """
        Keep frozen donor networks in eval mode even when the full model is set to train().
        """
        super().train(mode)

        if self.freeze_donors:
            self.resnet50.eval()
            self.vgg16bn.eval()

        return self

    def _resnet50_feats(self, x: torch.Tensor):
        """
        ResNet50 donor feature up to layer2.
        """
        r = self.resnet50.conv1(x)
        r = self.resnet50.bn1(r)
        r = self.resnet50.relu(r)
        r = self.resnet50.maxpool(r)

        r = self.resnet50.layer1(r)
        r = self.resnet50.layer2(r)

        return r

    def _vgg16_feats(self, x: torch.Tensor):
        """
        VGG16-BN donor early feature.
        """
        v = self.vgg16bn.features[:14](x)
        return v

    def _hybrid_feats(self, x: torch.Tensor):
        """
        Donor features + trainable 1x1 projections.
        """
        if self.freeze_donors:
            with torch.no_grad():
                r = self._resnet50_feats(x)
                v = self._vgg16_feats(x)
        else:
            r = self._resnet50_feats(x)
            v = self._vgg16_feats(x)

        # Projection layers remain trainable.
        r = self.proj_res(r)
        v = self.proj_vgg(v)

        return r, v

    def _inception_stem_to_pre5b(self, x: torch.Tensor):
        """
        InceptionV3 stem up to pre-Mixed_5b.
        """
        x = self.inception.Conv2d_1a_3x3(x)
        x = self.inception.Conv2d_2a_3x3(x)
        x = self.inception.Conv2d_2b_3x3(x)
        x = self.inception.maxpool1(x)

        x = self.inception.Conv2d_3b_1x1(x)
        x = self.inception.Conv2d_4a_3x3(x)
        x = self.inception.maxpool2(x)

        return x

    def _inception_tail_from_5b(self, x: torch.Tensor):
        """
        InceptionV3 tail from Mixed_5b to Mixed_7c.
        """
        x = self.inception.Mixed_5b(x)
        x = self.inception.Mixed_5c(x)
        x = self.inception.Mixed_5d(x)

        x = self.inception.Mixed_6a(x)
        x = self.inception.Mixed_6b(x)
        x = self.inception.Mixed_6c(x)
        x = self.inception.Mixed_6d(x)
        x = self.inception.Mixed_6e(x)

        x = self.inception.Mixed_7a(x)
        x = self.inception.Mixed_7b(x)
        x = self.inception.Mixed_7c(x)

        return x

    def forward(self, x: torch.Tensor):
        """
        Input:
            x: [B, 3, 299, 299]

        Returns:
            emb128   : [B, 128]
            emb64    : [B, 64]
            feat_map : [B, 2048, 8, 8] -> used for Nakagami CAM head
        """

        # ----- Optional DCIEN preprocessing -----
        if self.use_prep:
            prep = self.preprocessor(x)
            prep = (prep - self.mean) / (self.std + 1e-8)

            x_in = torch.cat([x[:, 0:2, :, :], prep], dim=1)
        else:
            x_in = x

        # ----- Inception stem -----
        inc_pre5b = self._inception_stem_to_pre5b(x_in)

        # ----- Donor features -----
        r, v = self._hybrid_feats(x_in)

        # ----- Resize donor features and fuse -----
        r = F.interpolate(
            r,
            size=inc_pre5b.shape[-2:],
            mode="bilinear",
            align_corners=False
        )

        v = F.interpolate(
            v,
            size=inc_pre5b.shape[-2:],
            mode="bilinear",
            align_corners=False
        )

        fused = self.fuse_bn(inc_pre5b + r + v)

        # ----- Inception tail -----
        feat_map = self._inception_tail_from_5b(fused)
        feat_map = self.se_final(feat_map)

        # ----- Compact embedding head -----
        z = self.avgpool(feat_map).flatten(1)
        z = self.dropout(z)

        z = self.act(self.fc1(z))
        emb128 = self.act(self.fc2(z))
        emb64 = self.fc3(emb128)

        return emb128, emb64, feat_map

In [ ]:
# ===============================
# Cell 8: ADCE Module
# ===============================

import torch
import torch.nn as nn
import torch.nn.functional as F


class ADCE(nn.Module):
    """
    Attention-Driven Context Enhancement (ADCE)
    -------------------------------------------------------
    Input : [B, C, H, W]

    Steps:
      1) Channel attention using GAP + GMP + shared MLP.
      2) Gate input feature map.
      3) Concatenate original and gated feature maps.
      4) 1x1 conv compresses 2C -> C.
      5) Multi-dilated 3x3 convolution branches.
      6) Concatenate branch outputs.
      7) 1x1 conv reduces to out_channels.

    Output:
      [B, out_channels, H, W]

    Default:
      out_channels = C // 2
    """

    def __init__(
        self,
        in_channels: int,
        reduction: int = 8,
        dilations=(1, 2, 4, 6),
        out_channels: int = None,
        dropout_p: float = 0.0
    ):
        super().__init__()

        C = int(in_channels)
        hidden = max(C // reduction, 1)

        if out_channels is None:
            out_channels = max(C // 2, 1)

        self.in_channels = C
        self.out_channels = out_channels
        self.dilations = dilations

        # ----- Channel Attention: GAP + GMP -> shared MLP -----
        self.gap = nn.AdaptiveAvgPool2d(1)
        self.gmp = nn.AdaptiveMaxPool2d(1)

        self.ca_mlp = nn.Sequential(
            nn.Linear(C, hidden, bias=True),
            nn.ReLU(inplace=True),
            nn.Linear(hidden, C, bias=True)
        )

        self.ca_sigmoid = nn.Sigmoid()

        # ----- Fuse original and gated features -----
        self.fuse1x1 = nn.Sequential(
            nn.Conv2d(2 * C, C, kernel_size=1, bias=False),
            nn.BatchNorm2d(C),
            nn.ReLU(inplace=True)
        )

        # ----- Multi-dilated context branches -----
        self.branches = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(
                    C,
                    C,
                    kernel_size=3,
                    padding=d,
                    dilation=d,
                    bias=False
                ),
                nn.BatchNorm2d(C),
                nn.ReLU(inplace=True)
            )
            for d in dilations
        ])

        fused_C = C * len(dilations)

        # ----- Optional dropout before final reduction -----
        self.dropout = nn.Dropout2d(p=dropout_p) if dropout_p > 0 else nn.Identity()

        # ----- Reduce concatenated multi-scale features -----
        self.out = nn.Sequential(
            nn.Conv2d(fused_C, out_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x: torch.Tensor):
        """
        x: [B, C, H, W]
        """

        B, C, H, W = x.shape

        if C != self.in_channels:
            raise ValueError(
                f"ADCE expected {self.in_channels} input channels, but got {C}."
            )

        # ----- Channel attention gate -----
        avg = self.gap(x).view(B, C)
        mx = self.gmp(x).view(B, C)

        gate = self.ca_mlp(avg) + self.ca_mlp(mx)
        gate = self.ca_sigmoid(gate).view(B, C, 1, 1)

        x_gated = x * gate

        # ----- Original + gated fusion -----
        z = torch.cat([x, x_gated], dim=1)
        z = self.fuse1x1(z)

        # ----- Multi-dilated context aggregation -----
        branch_outputs = [branch(z) for branch in self.branches]
        z = torch.cat(branch_outputs, dim=1)

        # ----- Dropout + output reduction -----
        z = self.dropout(z)
        z = self.out(z)

        return z

In [ ]:
# ===============================
# Cell 9: Tri-modal Fusion + Final Stage-1 Model
# ===============================

import torch
import torch.nn as nn
import torch.nn.functional as F


# --------- small helper: per-modality (128,64) -> fused 128 with self-attention gates ----------
class ModalitySelfFuse(nn.Module):
    """
    Per-modality self-fusion.

    Input:
        e128: [B, 128]
        e64 : [B, 64]

    Process:
        LN(128), LN(64)
        Project 64 -> 128
        Get two scalar scores
        Softmax over the two paths
        Weighted sum -> 128-D feature

    Output:
        fused: [B, 128]
        w    : [B, 2]
    """

    def __init__(self, dim128=128, dim64=64):
        super().__init__()

        self.ln128 = nn.LayerNorm(dim128)
        self.ln64 = nn.LayerNorm(dim64)

        self.proj64to128 = nn.Linear(dim64, dim128, bias=True)

        self.score128 = nn.Linear(dim128, 1, bias=True)
        self.score64 = nn.Linear(dim128, 1, bias=True)

    def forward(self, e128, e64):
        e128n = self.ln128(e128)
        e64n = self.ln64(e64)

        e64u = self.proj64to128(e64n)

        s128 = self.score128(e128n)
        s64 = self.score64(e64u)

        w = torch.softmax(torch.cat([s128, s64], dim=1), dim=1)

        fused = w[:, :1] * e128n + w[:, 1:] * e64u

        return fused, w


# --------- cross-modal softmax weighting over three 128-D streams ----------
class CrossModalSoftmax(nn.Module):
    """
    Cross-modal softmax weighting over B-mode, H-scan, Nakagami streams.

    Input:
        fb, fh, fn: each [B, 128]

    Output:
        Fw: [B, 3, 128]
        W : [B, 3]
    """

    def __init__(self, dim=128):
        super().__init__()

        self.ln = nn.LayerNorm(dim)
        self.score = nn.Linear(dim, 1, bias=True)

    def forward(self, fb, fh, fn):
        fb = self.ln(fb)
        fh = self.ln(fh)
        fn = self.ln(fn)

        scores = torch.cat(
            [
                self.score(fb),
                self.score(fh),
                self.score(fn)
            ],
            dim=1
        )

        weights = torch.softmax(scores, dim=1)  # [B, 3]

        features = torch.stack([fb, fh, fn], dim=1)  # [B, 3, 128]
        weighted_features = weights.unsqueeze(-1) * features

        return weighted_features, weights


# --------- fusion block ----------
class TriModalFusion(nn.Module):
    """
    Tri-modal fusion block.

    Inputs:
        b128, b64, h128, h64, n128, n64

    Pipeline:
        1. Per-modality self-fusion: (128,64) -> 128
        2. Cross-modal softmax weighting: [B,3,128]
        3. Stack to pseudo feature map: [B,128,1,3]
        4. ADCE refinement

    Output:
        Z: [B, adce_out_channels, 1, 3]
        weights dictionary
    """

    def __init__(self, adce_out_channels=512):
        super().__init__()

        self.self_b = ModalitySelfFuse(128, 64)
        self.self_h = ModalitySelfFuse(128, 64)
        self.self_n = ModalitySelfFuse(128, 64)

        self.cross = CrossModalSoftmax(128)

        self.ln_after = nn.LayerNorm(128)

        self.adce = ADCE(
            in_channels=128,
            out_channels=adce_out_channels,
            reduction=8,
            dilations=(1, 2, 4, 6)
        )

    def forward(self, b128, b64, h128, h64, n128, n64):
        # 1) Per-modality self-fusion
        fb, wb = self.self_b(b128, b64)
        fh, wh = self.self_h(h128, h64)
        fn, wn = self.self_n(n128, n64)

        # 2) Cross-modal softmax weighting
        Fw, Wm = self.cross(fb, fh, fn)  # [B, 3, 128], [B, 3]

        # 3) Stack as pseudo 2D feature map for ADCE
        Fw = self.ln_after(Fw)
        Fw = Fw.transpose(1, 2)   # [B, 128, 3]
        Fw = Fw.unsqueeze(2)      # [B, 128, 1, 3]

        # 4) ADCE
        Z = self.adce(Fw)         # [B, adce_out_channels, 1, 3]

        return Z, {
            "self_w": {
                "bmode": wb,
                "hscan": wh,
                "nakagami": wn
            },
            "modal_w": Wm
        }


# --------- final end-to-end model ----------
class TriModalStage1(nn.Module):
    """
    Final tri-modal model.

    Branches:
        B-mode branch
        H-scan branch
        Nakagami branch

    Added outputs for new losses:
        1. Multi-CAM:
            CAM heads from branch feat_maps

        2. F-QUS:
            B-mode emb128 -> 8 QUS values

        3. P-QUS:
            final fused 64-D feature -> 1 logit
    """

    def __init__(
        self,
        adce_out_channels=512,
        num_classes=2,
        dropout_head=0.2,
        qus_dim=8
    ):
        super().__init__()

        # ----- three modality branches -----
        self.b_branch = PTLCNN_BMODE()
        self.h_branch = PTLCNN_HSCAN()
        self.n_branch = PTLCNN_NAKAGAMI()

        # ----- fusion -----
        self.fusion = TriModalFusion(adce_out_channels=adce_out_channels)

        # ----- classifier head -----
        self.gap = nn.AdaptiveAvgPool2d((1, 1))

        self.fc_fuse1 = nn.Linear(adce_out_channels, 128)
        self.fc_fuse2 = nn.Linear(128, 64)

        self.bn64 = nn.BatchNorm1d(64)
        self.drop = nn.Dropout(p=dropout_head)

        self.classifier = nn.Linear(64, num_classes)

        self.act = nn.ReLU(inplace=True)

        # ============================================================
        # New heads for auxiliary losses
        # ============================================================

        # ----- Multi-CAM heads -----
        # Input feat_map from each branch: [B, 2048, 8, 8]
        # Output CAM probability map: [B, 1, 8, 8]
        self.cam_b = nn.Sequential(
            nn.Conv2d(2048, 1, kernel_size=1, bias=True),
            nn.Sigmoid()
        )

        self.cam_h = nn.Sequential(
            nn.Conv2d(2048, 1, kernel_size=1, bias=True),
            nn.Sigmoid()
        )

        self.cam_n = nn.Sequential(
            nn.Conv2d(2048, 1, kernel_size=1, bias=True),
            nn.Sigmoid()
        )

        # ----- F-QUS head -----
        # B-mode feature predicts handcrafted QUS descriptors.
        # Because QUS targets will be StandardScaler-normalized,
        # final layer should be linear, not sigmoid.
        self.fqus_head = nn.Sequential(
            nn.Linear(128, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.2),
            nn.Linear(64, qus_dim)
        )

        # ----- P-QUS head -----
        # Predict probability of being clinically relevant over QUS threshold.
        # Output is raw logit; use BCEWithLogitsLoss during training.
        self.pqus_head = nn.Sequential(
            nn.Linear(64, 32),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.2),
            nn.Linear(32, 1)
        )

    def forward(self, xb, xh, xn):
        """
        Inputs:
            xb: B-mode image      [B, 3, 299, 299]
            xh: H-scan image      [B, 3, 299, 299]
            xn: Nakagami image    [B, 3, 299, 299]

        Returns:
            logits: [B, num_classes]

            output_dict contains:
                emb
                feat_maps
                cams
                qus_pred
                pqus_logit
                weights
        """

        # ============================================================
        # 1. Branch forward
        # ============================================================
        b128, b64, b_featmap = self.b_branch(xb)
        h128, h64, h_featmap = self.h_branch(xh)
        n128, n64, n_featmap = self.n_branch(xn)

        # ============================================================
        # 2. Multi-CAM maps
        # ============================================================
        cam_b = self.cam_b(b_featmap)  # [B, 1, 8, 8]
        cam_h = self.cam_h(h_featmap)  # [B, 1, 8, 8]
        cam_n = self.cam_n(n_featmap)  # [B, 1, 8, 8]

        cams = {
            "bmode": cam_b,
            "hscan": cam_h,
            "nakagami": cam_n
        }

        # ============================================================
        # 3. F-QUS prediction from B-mode branch
        # ============================================================
        qus_pred = self.fqus_head(b128)  # [B, 8]

        # ============================================================
        # 4. Tri-modal fusion
        # ============================================================
        Z, wdict = self.fusion(
            b128, b64,
            h128, h64,
            n128, n64
        )  # Z: [B, 512, 1, 3]

        # ============================================================
        # 5. Final classifier head
        # ============================================================
        z = self.gap(Z).flatten(1)      # [B, adce_out_channels]

        z = self.act(self.fc_fuse1(z))  # [B, 128]
        z = self.act(self.fc_fuse2(z))  # [B, 64]

        z = self.bn64(z)

        # Use this 64-D feature for P-QUS before dropout.
        fused64 = z

        pqus_logit = self.pqus_head(fused64)  # [B, 1]

        z_drop = self.drop(fused64)
        logits = self.classifier(z_drop)      # [B, num_classes]

        return logits, {
            "emb": {
                "b128": b128,
                "h128": h128,
                "n128": n128,
                "b64": b64,
                "h64": h64,
                "n64": n64,
                "fused64": fused64
            },
            "feat_maps": {
                "bmode": b_featmap,
                "hscan": h_featmap,
                "nakagami": n_featmap
            },
            "cams": cams,
            "qus_pred": qus_pred,
            "pqus_logit": pqus_logit,
            "weights": wdict
        }

In [ ]:
# ===============================
# Cell 10: Build Master Index for Tri-modal BUET-BUSD Dataset
# ===============================

import os
import glob
import pandas as pd

# Root path
ROOT = "/kaggle/input/updated-final-boss/Updated_Final(Registed)Dataset/BUET_BUSD/Binary"

LABEL_MAP = {
    "Benign": 0,
    "Malignant": 1
}

FOLDS = [f"Fold{i}" for i in range(1, 6)]


def build_master_index(root=ROOT, require_mask=True, verbose=True):
    """
    Build master dataframe containing aligned paths for:
        Machine B-mode
        H-scan
        Nakagami
        Mask

    B-mode is used as the canonical file list.

    Returns:
        MASTER_DF with columns:
            fold, label_name, label, fname,
            b_path, h_path, n_path, mask_path
    """

    rows = []
    skipped = []

    for fold in FOLDS:
        for lbl_name, lbl_id in LABEL_MAP.items():

            b_dir = os.path.join(root, "Machine_B-mode", fold, lbl_name)

            if not os.path.exists(b_dir):
                print(f"[Warning] Missing B-mode directory: {b_dir}")
                continue

            b_paths = sorted(glob.glob(os.path.join(b_dir, "*.png")))

            for b_path in b_paths:
                fname = os.path.basename(b_path)

                rel = os.path.join(fold, lbl_name, fname)

                h_path = os.path.join(root, "H-scan", rel)
                n_path = os.path.join(root, "Nakagami", rel)
                m_path = os.path.join(root, "Mask", rel)

                exists_b = os.path.exists(b_path)
                exists_h = os.path.exists(h_path)
                exists_n = os.path.exists(n_path)
                exists_m = os.path.exists(m_path)

                if not (exists_b and exists_h and exists_n):
                    skipped.append({
                        "fold": fold,
                        "label_name": lbl_name,
                        "fname": fname,
                        "reason": "missing modality",
                        "b_exists": exists_b,
                        "h_exists": exists_h,
                        "n_exists": exists_n,
                        "mask_exists": exists_m
                    })
                    continue

                if require_mask and not exists_m:
                    skipped.append({
                        "fold": fold,
                        "label_name": lbl_name,
                        "fname": fname,
                        "reason": "missing mask",
                        "b_exists": exists_b,
                        "h_exists": exists_h,
                        "n_exists": exists_n,
                        "mask_exists": exists_m
                    })
                    continue

                rows.append({
                    "fold": fold,
                    "label_name": lbl_name,
                    "label": lbl_id,
                    "fname": fname,
                    "b_path": b_path,
                    "h_path": h_path,
                    "n_path": n_path,
                    "mask_path": m_path if exists_m else ""
                })

    df = pd.DataFrame(rows)

    if len(df) > 0:
        df = df.sort_values(["fold", "label", "fname"]).reset_index(drop=True)

    skipped_df = pd.DataFrame(skipped)

    if verbose:
        print("=" * 60)
        print("Master index summary")
        print("=" * 60)

        if len(df) > 0:
            print(df.groupby(["fold", "label_name"]).size())
        else:
            print("[Warning] MASTER_DF is empty.")

        print("\nTotal usable samples:", len(df))
        print("Total skipped samples:", len(skipped_df))

        if len(skipped_df) > 0:
            print("\nSkipped reason counts:")
            print(skipped_df["reason"].value_counts())

    return df, skipped_df


MASTER_DF, SKIPPED_DF = build_master_index(
    root=ROOT,
    require_mask=True,
    verbose=True
)

MASTER_DF.head()

Master index summary
fold   label_name
Fold1  Benign        29
       Malignant     11
Fold2  Benign        29
       Malignant     11
Fold3  Benign        28
       Malignant     11
Fold4  Benign        28
       Malignant     11
Fold5  Benign        28
       Malignant     11
dtype: int64

Total usable samples: 197
Total skipped samples: 0


,fold,label_name,label,fname,b_path,h_path,n_path,mask_path
0,Fold1,Benign,0,09-11-49.png,/kaggle/input/updated-final-boss/Updated_Final...,/kaggle/input/updated-final-boss/Updated_Final...,/kaggle/input/updated-final-boss/Updated_Final...,/kaggle/input/updated-final-boss/Updated_Final...
1,Fold1,Benign,0,09-14-46.png,/kaggle/input/updated-final-boss/Updated_Final...,/kaggle/input/updated-final-boss/Updated_Final...,/kaggle/input/updated-final-boss/Updated_Final...,/kaggle/input/updated-final-boss/Updated_Final...
2,Fold1,Benign,0,09-23-24.png,/kaggle/input/updated-final-boss/Updated_Final...,/kaggle/input/updated-final-boss/Updated_Final...,/kaggle/input/updated-final-boss/Updated_Final...,/kaggle/input/updated-final-boss/Updated_Final...
3,Fold1,Benign,0,09-31-49.png,/kaggle/input/updated-final-boss/Updated_Final...,/kaggle/input/updated-final-boss/Updated_Final...,/kaggle/input/updated-final-boss/Updated_Final...,/kaggle/input/updated-final-boss/Updated_Final...
4,Fold1,Benign,0,09-49-27.png,/kaggle/input/updated-final-boss/Updated_Final...,/kaggle/input/updated-final-boss/Updated_Final...,/kaggle/input/updated-final-boss/Updated_Final...,/kaggle/input/updated-final-boss/Updated_Final...


In [ ]:
# ===============================
# Cell 11: Generate / Load Raw QUS Feature CSV
# ===============================
# Important:
#   This cell calculates RAW QUS features only.
#   Do NOT normalize here.
#   Do NOT train SVM here.
#
# Fold-wise normalization + fold-wise SVM will be done later inside run_fold().
# This avoids cross-validation leakage.
# ===============================

QUS_RAW_CSV_PATH = "/kaggle/working/buet_busd_raw_qus_features.csv"


def compute_raw_qus_dataframe(master_df, csv_path=QUS_RAW_CSV_PATH, force_recompute=False):
    """
    Calculate raw QUS features for every sample in MASTER_DF.

    Uses:
        b_path    -> grayscale B-mode image
        mask_path -> binary mask

    Saves:
        /kaggle/working/buet_busd_raw_qus_features.csv

    Returns:
        qus_df
    """

    if os.path.exists(csv_path) and not force_recompute:
        print(f"[QUS] Loading existing raw QUS CSV: {csv_path}")
        qus_df = pd.read_csv(csv_path)
        return qus_df

    print("[QUS] Computing raw QUS features...")
    print("[QUS] This may take some time on first run.")

    rows = []

    for _, row in tqdm(master_df.iterrows(), total=len(master_df), desc="Computing QUS"):
        b_path = row["b_path"]
        mask_path = row["mask_path"]

        image = cv2.imread(b_path, cv2.IMREAD_GRAYSCALE)
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)

        status = "ok"

        try:
            if image is None:
                raise ValueError(f"Could not read image: {b_path}")

            if mask is None:
                raise ValueError(f"Could not read mask: {mask_path}")

            qus_dict = quantitative_ultrasound_features_dict(image, mask)

        except Exception as e:
            status = f"failed: {str(e)}"
            qus_dict = {name: 0.0 for name in QUS_FEATURE_NAMES}

        out_row = {
            "fold": row["fold"],
            "label_name": row["label_name"],
            "label": int(row["label"]),
            "fname": row["fname"],
            "b_path": b_path,
            "mask_path": mask_path,
            "qus_status": status
        }

        out_row.update(qus_dict)
        rows.append(out_row)

    qus_df = pd.DataFrame(rows)

    qus_df.to_csv(csv_path, index=False)
    print(f"[QUS] Saved raw QUS CSV to: {csv_path}")

    return qus_df


def merge_raw_qus_into_master(master_df, qus_df):
    """
    Merge raw QUS columns into MASTER_DF.

    Merge key:
        fold, label_name, label, fname
    """

    merge_keys = ["fold", "label_name", "label", "fname"]

    keep_cols = merge_keys + ["qus_status"] + QUS_FEATURE_NAMES

    qus_small = qus_df[keep_cols].copy()

    merged_df = master_df.merge(
        qus_small,
        on=merge_keys,
        how="left"
    )

    # Safety: replace missing QUS values with 0
    for col in QUS_FEATURE_NAMES:
        if col not in merged_df.columns:
            merged_df[col] = 0.0

        merged_df[col] = merged_df[col].fillna(0.0).astype(np.float32)

    if "qus_status" not in merged_df.columns:
        merged_df["qus_status"] = "missing"
    else:
        merged_df["qus_status"] = merged_df["qus_status"].fillna("missing")

    return merged_df


# Compute or load raw QUS CSV
RAW_QUS_DF = compute_raw_qus_dataframe(
    MASTER_DF,
    csv_path=QUS_RAW_CSV_PATH,
    force_recompute=False
)

# Merge raw QUS features into MASTER_DF
MASTER_DF = merge_raw_qus_into_master(MASTER_DF, RAW_QUS_DF)

print("=" * 60)
print("MASTER_DF after raw QUS merge")
print("=" * 60)
print("Shape:", MASTER_DF.shape)
print("\nQUS status counts:")
print(MASTER_DF["qus_status"].value_counts())

display_cols = ["fold", "label_name", "fname", "label"] + QUS_FEATURE_NAMES
MASTER_DF[display_cols].head()

[QUS] Computing raw QUS features...
[QUS] This may take some time on first run.


Computing QUS: 100%|██████████| 197/197 [00:14<00:00, 13.95it/s]

[QUS] Saved raw QUS CSV to: /kaggle/working/buet_busd_raw_qus_features.csv
MASTER_DF after raw QUS merge
Shape: (197, 17)

QUS status counts:
qus_status
ok    197
Name: count, dtype: int64


,fold,label_name,fname,label,ED,SAF,TH,ASR,CVX,SOLIDITY,CCONTRAST,LBD
0,Fold1,Benign,09-11-49.png,0,0.951279,0.171509,27.458942,0.567251,0.736154,0.886532,7.262106,25.660164
1,Fold1,Benign,09-14-46.png,0,0.965844,0.213129,45.175194,0.750890,0.909921,0.928355,4.411880,4.742825
2,Fold1,Benign,09-23-24.png,0,0.920169,0.114931,34.373260,0.688312,0.886125,0.943798,27.274866,8.285210
3,Fold1,Benign,09-31-49.png,0,0.960437,0.216331,23.598499,0.504587,0.613258,0.905958,7.962976,45.221306
4,Fold1,Benign,09-49-27.png,0,0.954579,0.115384,14.798069,0.510417,0.805902,0.962218,1.106703,36.191181


In [ ]:
# ===============================
# Cell 12: Dataset, Joint Augmentation, and Loader Factory
# ===============================

import random
import numpy as np
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF


# Normalized QUS columns will be created fold-wise 
QUS_NORM_FEATURE_NAMES = [f"{name}_N" for name in QUS_FEATURE_NAMES]


# ------- Joint paired augmentations so all three modalities + mask get identical ops --------
class JointRandAug:
    def __init__(self, img_size=299, enable=True):
        self.enable = enable
        self.img_size = img_size

    def _sample(self):
        if not self.enable:
            return {
                "angle": 0.0,
                "flip_h": False,
                "flip_v": False,
                "scale": 1.0,
                "tx": 0,
                "ty": 0
            }

        angle = random.uniform(-10, 10)
        flip_h = random.random() < 0.5
        flip_v = random.random() < 0.1

        scale = random.uniform(0.95, 1.05)

        max_t = int(0.05 * self.img_size)
        tx = random.randint(-max_t, max_t)
        ty = random.randint(-max_t, max_t)

        return {
            "angle": angle,
            "flip_h": flip_h,
            "flip_v": flip_v,
            "scale": scale,
            "tx": tx,
            "ty": ty
        }

    def _apply_one(self, img, params, is_mask=False):
        interpolation = (
            TF.InterpolationMode.NEAREST
            if is_mask
            else TF.InterpolationMode.BILINEAR
        )

        img = TF.resize(
            img,
            [self.img_size, self.img_size],
            interpolation=interpolation
        )

        if params["flip_h"]:
            img = TF.hflip(img)

        if params["flip_v"]:
            img = TF.vflip(img)

        img = TF.affine(
            img,
            angle=params["angle"],
            translate=(params["tx"], params["ty"]),
            scale=params["scale"],
            shear=[0.0, 0.0],
            interpolation=interpolation,
            fill=0
        )

        return img

    def __call__(self, b_img, h_img, n_img, m_img=None):
        params = self._sample()

        b_img = self._apply_one(b_img, params, is_mask=False)
        h_img = self._apply_one(h_img, params, is_mask=False)
        n_img = self._apply_one(n_img, params, is_mask=False)

        if m_img is not None:
            m_img = self._apply_one(m_img, params, is_mask=True)

        return b_img, h_img, n_img, m_img


# ------- Dataset --------
class TriModalBUETBUSDataset(Dataset):
    """
    Dataset for tri-modal BUET-BUSD binary classification.

    Returns by default:
        b, h, n, y

    If return_aux=True:
        b, h, n, y, mask, qus_vector, svm_qus_prob

    Shapes:
        b/h/n       : [3, 299, 299]
        y           : scalar long
        mask        : [1, 299, 299], float 0/1
        qus_vector  : [8], float
        svm_prob    : [1], float
    """

    def __init__(
        self,
        df,
        img_size=299,
        augment=False,
        return_aux=False
    ):
        self.df = df.reset_index(drop=True)
        self.img_size = img_size
        self.aug = JointRandAug(img_size=img_size, enable=augment)
        self.return_aux = return_aux

        if self.return_aux:
            missing_cols = []

            for col in QUS_NORM_FEATURE_NAMES:
                if col not in self.df.columns:
                    missing_cols.append(col)

            if "svm_qus_prob" not in self.df.columns:
                missing_cols.append("svm_qus_prob")

            if "mask_path" not in self.df.columns:
                missing_cols.append("mask_path")

            if len(missing_cols) > 0:
                raise ValueError(
                    f"Dataset return_aux=True but missing columns: {missing_cols}"
                )

    def _read_rgb(self, path):
        return Image.open(path).convert("RGB")

    def _read_mask(self, path):
        mask = Image.open(path).convert("L")
        return mask

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        row = self.df.iloc[i]

        b = self._read_rgb(row.b_path)
        h = self._read_rgb(row.h_path)
        n = self._read_rgb(row.n_path)

        m = None
        if self.return_aux:
            m = self._read_mask(row.mask_path)

        # Apply same geometric augmentation to all modalities and mask
        b, h, n, m = self.aug(b, h, n, m)

        # Images to tensors in [0,1]
        b = TF.to_tensor(b)
        h = TF.to_tensor(h)
        n = TF.to_tensor(n)

        y = torch.tensor(int(row.label), dtype=torch.long)

        if not self.return_aux:
            return b, h, n, y

        # Mask to float [1,H,W], binary 0/1
        mask = TF.pil_to_tensor(m).float()  # [1,H,W], usually 0/255 or 0/1
        if mask.max() > 1:
            mask = mask / 255.0
        mask = (mask > 0.5).float()

        # Normalized QUS vector, created fold-wise in Cell 13A
        qus_vector = torch.tensor(
            row[QUS_NORM_FEATURE_NAMES].values.astype(np.float32),
            dtype=torch.float32
        )

        # Fixed QUS-SVM probability target for P-QUS
        svm_qus_prob = torch.tensor(
            [float(row.svm_qus_prob)],
            dtype=torch.float32
        )

        return b, h, n, y, mask, qus_vector, svm_qus_prob


# Keep old class name as alias, so older code does not immediately break
TriModalOASBUDDataset = TriModalBUETBUSDataset


# ------- Loader factory for a given fold --------
def make_fold_loaders(
    fold_df,
    val_fold="Fold1",
    batch_size=8,
    img_size=299,
    num_workers=2,
    pin_memory=True,
    return_aux=True
):
    """
    Creates train and validation loaders.

    Important:
        fold_df should already contain:
            normalized QUS columns
            svm_qus_prob

        This will be prepared by Cell 13A.
    """

    train_df = fold_df[fold_df.fold != val_fold].copy()
    val_df = fold_df[fold_df.fold == val_fold].copy()

    train_ds = TriModalBUETBUSDataset(
        train_df,
        img_size=img_size,
        augment=True,
        return_aux=return_aux
    )

    val_ds = TriModalBUETBUSDataset(
        val_df,
        img_size=img_size,
        augment=False,
        return_aux=return_aux
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=pin_memory,
        drop_last=True
    )

    val_loader = DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=pin_memory,
        drop_last=False
    )

    print(f"[Fold {val_fold}] Train: {len(train_ds)}  Val: {len(val_ds)}")

    if return_aux:
        print("[Loader] Returning: b, h, n, y, mask, qus_vector, svm_qus_prob")
    else:
        print("[Loader] Returning: b, h, n, y")

    return train_loader, val_loader

In [ ]:
# ===============================
# Cell 14: Fold-wise QUS Normalization + QUS-SVM Teacher
# ===============================
# Important:
#   Fit StandardScaler using TRAIN folds only.
#   Fit SVM using TRAIN folds only.
#   Then transform/predict both train and val.
#
# This avoids data leakage.
# ===============================

from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC


def prepare_qus_for_fold(
    master_df,
    val_fold="Fold1",
    svm_kernel="rbf",
    svm_C=1.0,
    svm_gamma="scale",
    svm_probability=True,
    verbose=True
):
    """
    Prepare fold-specific dataframe with:
        1. normalized QUS columns: ED_N, SAF_N, ...
        2. fixed SVM-QUS probability: svm_qus_prob

    Returns:
        fold_df, qus_scaler, qus_svm
    """

    df = master_df.copy().reset_index(drop=True)

    # Safety check
    missing_raw = [col for col in QUS_FEATURE_NAMES if col not in df.columns]
    if len(missing_raw) > 0:
        raise ValueError(
            f"MASTER_DF is missing raw QUS columns: {missing_raw}. "
            "Run Cell 12A first."
        )

    train_mask = df["fold"] != val_fold
    val_mask = df["fold"] == val_fold

    train_df = df[train_mask].copy()
    val_df = df[val_mask].copy()

    if len(train_df) == 0 or len(val_df) == 0:
        raise ValueError(f"Invalid fold split for val_fold={val_fold}.")

    X_train_raw = train_df[QUS_FEATURE_NAMES].values.astype(np.float32)
    y_train = train_df["label"].values.astype(int)

    X_all_raw = df[QUS_FEATURE_NAMES].values.astype(np.float32)

    # Replace NaN/Inf safely
    X_train_raw = np.nan_to_num(X_train_raw, nan=0.0, posinf=0.0, neginf=0.0)
    X_all_raw = np.nan_to_num(X_all_raw, nan=0.0, posinf=0.0, neginf=0.0)

    # 1. Fit scaler on training folds only
    qus_scaler = StandardScaler()
    qus_scaler.fit(X_train_raw)

    X_all_norm = qus_scaler.transform(X_all_raw).astype(np.float32)
    X_train_norm = X_all_norm[train_mask.values]

    # Add normalized QUS columns
    for j, col in enumerate(QUS_NORM_FEATURE_NAMES):
        df[col] = X_all_norm[:, j].astype(np.float32)

    # 2. Fit SVM on normalized train QUS only
    qus_svm = SVC(
        kernel=svm_kernel,
        C=svm_C,
        gamma=svm_gamma,
        probability=svm_probability,
        class_weight="balanced",
        random_state=SEED
    )

    qus_svm.fit(X_train_norm, y_train)

    # 3. Predict probability for all samples using fold-trained SVM
    if svm_probability:
        prob_all = qus_svm.predict_proba(X_all_norm)

        # Find malignant class index, label=1
        class_list = list(qus_svm.classes_)
        if 1 in class_list:
            malignant_idx = class_list.index(1)
        else:
            malignant_idx = 0

        svm_qus_prob = prob_all[:, malignant_idx]

    else:
        # fallback using decision function converted to pseudo-probability
        decision = qus_svm.decision_function(X_all_norm)
        svm_qus_prob = 1.0 / (1.0 + np.exp(-decision))

    df["svm_qus_prob"] = svm_qus_prob.astype(np.float32)

    if verbose:
        print("=" * 60)
        print(f"Fold-wise QUS preparation for validation fold: {val_fold}")
        print("=" * 60)

        print("Train samples:", int(train_mask.sum()))
        print("Val samples:", int(val_mask.sum()))

        print("\nSVM classes:", qus_svm.classes_)

        print("\nTrain svm_qus_prob summary:")
        print(df.loc[train_mask, "svm_qus_prob"].describe())

        print("\nVal svm_qus_prob summary:")
        print(df.loc[val_mask, "svm_qus_prob"].describe())

        print("\nNormalized QUS columns:")
        print(QUS_NORM_FEATURE_NAMES)

    return df, qus_scaler, qus_svm


# train_loader, val_loader = make_fold_loaders(MASTER_DF, val_fold="Fold1", batch_size=8, img_size=299)

FOLD_DF, QUS_SCALER, QUS_SVM = prepare_qus_for_fold(
    MASTER_DF,
    val_fold="Fold1",
    verbose=True
)

train_loader, val_loader = make_fold_loaders(
    FOLD_DF,
    val_fold="Fold1",
    batch_size=8,
    img_size=299,
    num_workers=2,
    return_aux=True
)

# Quick batch shape check
batch = next(iter(train_loader))
xb, xh, xn, y, mask, qus_vec, svm_prob = batch

print("\nBatch shape check:")
print("xb       :", xb.shape)
print("xh       :", xh.shape)
print("xn       :", xn.shape)
print("y        :", y.shape)
print("mask     :", mask.shape)
print("qus_vec  :", qus_vec.shape)
print("svm_prob :", svm_prob.shape)

Fold-wise QUS preparation for validation fold: Fold1
Train samples: 157
Val samples: 40

SVM classes: [0 1]

Train svm_qus_prob summary:
count    157.000000
mean       0.293096
std        0.369684
min        0.002617
25%        0.013949
50%        0.039268
75%        0.743624
max        0.973773
Name: svm_qus_prob, dtype: float64

Val svm_qus_prob summary:
count    40.000000
mean      0.277887
std       0.349642
min       0.004708
25%       0.014209
50%       0.069356
75%       0.573014
max       0.971994
Name: svm_qus_prob, dtype: float64

Normalized QUS columns:
['ED_N', 'SAF_N', 'TH_N', 'ASR_N', 'CVX_N', 'SOLIDITY_N', 'CCONTRAST_N', 'LBD_N']
[Fold Fold1] Train: 157  Val: 40
[Loader] Returning: b, h, n, y, mask, qus_vector, svm_qus_prob

Batch shape check:
xb       : torch.Size([8, 3, 299, 299])
xh       : torch.Size([8, 3, 299, 299])
xn       : torch.Size([8, 3, 299, 299])
y        : torch.Size([8])
mask     : torch.Size([8, 1, 299, 299])
qus_vec  : torch.Size([8, 8])
svm_prob : tor

In [ ]:
# ===============================
# Cell 14: Training / Validation with CCE + CAM + F-QUS + P-QUS
# ===============================

import os
import time
import torch
import torch.nn as nn
from sklearn.metrics import f1_score, confusion_matrix
import joblib


EPOCHS = 50
CKPT_DIR = "/kaggle/working"


# -------------------------------
# Default loss weights
# -------------------------------

ALPHA_CAM = 0.30
BETA_FQUS = 0.03
BETA_PQUS = 0.01


def unpack_batch(batch, device):
    """
    Supports both old and new batch formats.

    Old:
        b, h, n, y

    New:
        b, h, n, y, mask, qus_true, p_ml
    """

    if len(batch) == 4:
        xb, xh, xn, y = batch

        xb = xb.to(device, non_blocking=True)
        xh = xh.to(device, non_blocking=True)
        xn = xn.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        return xb, xh, xn, y, None, None, None

    elif len(batch) == 7:
        xb, xh, xn, y, mask, qus_true, p_ml = batch

        xb = xb.to(device, non_blocking=True)
        xh = xh.to(device, non_blocking=True)
        xn = xn.to(device, non_blocking=True)
        y = y.to(device, non_blocking=True)

        mask = mask.to(device, non_blocking=True).float()
        qus_true = qus_true.to(device, non_blocking=True).float()
        p_ml = p_ml.to(device, non_blocking=True).float()

        return xb, xh, xn, y, mask, qus_true, p_ml

    else:
        raise ValueError(f"Unexpected batch length: {len(batch)}")


def compute_total_loss(
    logits,
    out,
    y,
    mask,
    qus_true,
    p_ml,
    ce_loss_fn,
    cam_loss_fn,
    fqus_loss_fn,
    pqus_loss_fn,
    alpha_cam=ALPHA_CAM,
    beta_fqus=BETA_FQUS,
    beta_pqus=BETA_PQUS
):
    """
    Total loss:

        L = L_CCE + alpha * L_CAM + beta_f * L_FQUS + beta_p * L_PQUS

    If auxiliary targets are missing, only CCE is used.
    """

    loss_cls = ce_loss_fn(logits, y)

    loss_cam = logits.new_tensor(0.0)
    loss_fqus = logits.new_tensor(0.0)
    loss_pqus = logits.new_tensor(0.0)

    if isinstance(out, dict):
        if mask is not None and "cams" in out:
            loss_cam = cam_loss_fn(out["cams"], mask)

        if qus_true is not None and "qus_pred" in out:
            loss_fqus = fqus_loss_fn(out["qus_pred"], qus_true)

        if p_ml is not None and "pqus_logit" in out:
            loss_pqus = pqus_loss_fn(
                out["pqus_logit"].view(-1),
                p_ml.view(-1)
            )

    loss_total = (
        loss_cls
        + alpha_cam * loss_cam
        + beta_fqus * loss_fqus
        + beta_pqus * loss_pqus
    )

    loss_parts = {
        "total": loss_total.detach(),
        "cls": loss_cls.detach(),
        "cam": loss_cam.detach(),
        "fqus": loss_fqus.detach(),
        "pqus": loss_pqus.detach()
    }

    return loss_total, loss_parts


def train_one_epoch(
    model,
    loader,
    optimizer,
    ce_loss_fn,
    cam_loss_fn,
    fqus_loss_fn,
    pqus_loss_fn,
    device,
    alpha_cam=ALPHA_CAM,
    beta_fqus=BETA_FQUS,
    beta_pqus=BETA_PQUS,
    grad_clip=None
):
    model.train()

    running = {
        "total": 0.0,
        "cls": 0.0,
        "cam": 0.0,
        "fqus": 0.0,
        "pqus": 0.0
    }

    correct = 0
    total = 0

    for batch in loader:
        xb, xh, xn, y, mask, qus_true, p_ml = unpack_batch(batch, device)

        optimizer.zero_grad(set_to_none=True)

        model_out = model(xb, xh, xn)

        if isinstance(model_out, tuple):
            logits, out = model_out
        else:
            logits = model_out
            out = {}

        loss, loss_parts = compute_total_loss(
            logits=logits,
            out=out,
            y=y,
            mask=mask,
            qus_true=qus_true,
            p_ml=p_ml,
            ce_loss_fn=ce_loss_fn,
            cam_loss_fn=cam_loss_fn,
            fqus_loss_fn=fqus_loss_fn,
            pqus_loss_fn=pqus_loss_fn,
            alpha_cam=alpha_cam,
            beta_fqus=beta_fqus,
            beta_pqus=beta_pqus
        )

        loss.backward()

        if grad_clip is not None:
            torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)

        optimizer.step()

        bs = y.size(0)

        for k in running:
            running[k] += loss_parts[k].item() * bs

        preds = logits.argmax(dim=1)
        correct += (preds == y).sum().item()
        total += bs

    avg_losses = {
        k: running[k] / max(total, 1)
        for k in running
    }

    acc = correct / max(total, 1)

    return avg_losses, acc


@torch.no_grad()
def eval_one_epoch(
    model,
    loader,
    ce_loss_fn,
    cam_loss_fn,
    fqus_loss_fn,
    pqus_loss_fn,
    device,
    alpha_cam=ALPHA_CAM,
    beta_fqus=BETA_FQUS,
    beta_pqus=BETA_PQUS
):
    model.eval()

    running = {
        "total": 0.0,
        "cls": 0.0,
        "cam": 0.0,
        "fqus": 0.0,
        "pqus": 0.0
    }

    correct = 0
    total = 0

    all_preds = []
    all_truth = []

    for batch in loader:
        xb, xh, xn, y, mask, qus_true, p_ml = unpack_batch(batch, device)

        model_out = model(xb, xh, xn)

        if isinstance(model_out, tuple):
            logits, out = model_out
        else:
            logits = model_out
            out = {}

        loss, loss_parts = compute_total_loss(
            logits=logits,
            out=out,
            y=y,
            mask=mask,
            qus_true=qus_true,
            p_ml=p_ml,
            ce_loss_fn=ce_loss_fn,
            cam_loss_fn=cam_loss_fn,
            fqus_loss_fn=fqus_loss_fn,
            pqus_loss_fn=pqus_loss_fn,
            alpha_cam=alpha_cam,
            beta_fqus=beta_fqus,
            beta_pqus=beta_pqus
        )

        bs = y.size(0)

        for k in running:
            running[k] += loss_parts[k].item() * bs

        preds = logits.argmax(dim=1)
        correct += (preds == y).sum().item()
        total += bs

        all_preds.append(preds.detach().cpu())
        all_truth.append(y.detach().cpu())

    avg_losses = {
        k: running[k] / max(total, 1)
        for k in running
    }

    acc = correct / max(total, 1)

    all_preds = torch.cat(all_preds).numpy() if all_preds else []
    all_truth = torch.cat(all_truth).numpy() if all_truth else []

    return avg_losses, acc, all_truth, all_preds


def compute_final_metrics(y_true, y_pred):
    """
    Labels:
        0 = Benign
        1 = Malignant
    """

    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])

    if cm.size == 4:
        tn, fp, fn, tp = cm.ravel()
    else:
        tn, fp, fn, tp = 0, 0, 0, 0

    sens = tp / (tp + fn + 1e-8)
    spec = tn / (tn + fp + 1e-8)

    f1 = f1_score(
        y_true,
        y_pred,
        average="binary",
        zero_division=0
    )

    balanced_acc = 0.5 * (sens + spec)

    return sens, spec, f1, balanced_acc, cm


def get_checkpoint_score(save_best_by, va_acc, sens, spec, f1, balanced_acc):
    """
    Decide which metric controls checkpoint saving.
    """

    if save_best_by == "acc":
        return va_acc

    if save_best_by == "f1":
        return f1

    if save_best_by == "balanced_acc":
        return balanced_acc

    if save_best_by == "sens":
        return sens

    raise ValueError(
        "save_best_by must be one of: 'acc', 'f1', 'balanced_acc', 'sens'"
    )


def run_fold(
    val_fold="Fold1",
    batch_size=8,
    img_size=299,
    lr=3e-4,
    weight_decay=1e-4,
    num_workers=2,
    alpha_cam=ALPHA_CAM,
    beta_fqus=BETA_FQUS,
    beta_pqus=BETA_PQUS,
    grad_clip=None,
    ce_weight_malignant=None,
    save_best_by="balanced_acc"
):
    """
    Full fold training.

    Steps:
        1. Prepare fold-wise QUS scaler + SVM teacher.
        2. Create loaders returning images + mask + QUS + SVM probability.
        3. Train model with CCE + CAM + F-QUS + P-QUS.
        4. Save best checkpoint based on selected validation metric:
           acc / f1 / balanced_acc / sens.
    """

    print("\n" + "=" * 80)
    print(f"Starting fold: {val_fold}")
    print("=" * 80)

    # ------------------------------------------------------------
    # 1. Fold-wise QUS normalization + SVM teacher
    # ------------------------------------------------------------
    fold_df, qus_scaler, qus_svm = prepare_qus_for_fold(
        MASTER_DF,
        val_fold=val_fold,
        verbose=True
    )

    # Save fold-specific scaler/SVM for reproducibility
    scaler_path = os.path.join(CKPT_DIR, f"qus_scaler_{val_fold}.joblib")
    svm_path = os.path.join(CKPT_DIR, f"qus_svm_{val_fold}.joblib")

    try:
        joblib.dump(qus_scaler, scaler_path)
        joblib.dump(qus_svm, svm_path)
        print(f"[Saved] QUS scaler: {scaler_path}")
        print(f"[Saved] QUS SVM   : {svm_path}")
    except Exception as e:
        print(f"[Warning] Could not save QUS scaler/SVM: {e}")

    # ------------------------------------------------------------
    # 2. Loaders
    # ------------------------------------------------------------
    train_loader, val_loader = make_fold_loaders(
        fold_df,
        val_fold=val_fold,
        batch_size=batch_size,
        img_size=img_size,
        num_workers=num_workers,
        return_aux=True
    )

    # ------------------------------------------------------------
    # 3. Model / Loss / Optimizer
    # ------------------------------------------------------------
    model = TriModalStage1().to(device)

    # Class-weighted CE.
    # class 0 = Benign, class 1 = Malignant
    if ce_weight_malignant is not None:
        ce_weights = torch.tensor(
            [1.0, float(ce_weight_malignant)],
            dtype=torch.float32
        ).to(device)

        ce_loss_fn = nn.CrossEntropyLoss(weight=ce_weights)

        print(
            f"[CE] Using class weights: "
            f"benign=1.0, malignant={float(ce_weight_malignant):.3f}"
        )
    else:
        ce_loss_fn = nn.CrossEntropyLoss()
        print("[CE] Using normal CrossEntropyLoss")

    cam_loss_fn = MultiCAMLoss(
        lambda_neighbor=1.0,
        lambda_iou_fg=1.0,
        lambda_iou_bg=1.0,
        lambda_boundary=0.0,
        boundary_weight=10.0,
        apply_sigmoid=False,
        normalize_cam=False
    ).to(device)

    fqus_loss_fn = nn.MSELoss()
    pqus_loss_fn = nn.BCEWithLogitsLoss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    # ------------------------------------------------------------
    # 4. Train
    # ------------------------------------------------------------
    best_score = -1.0
    best_acc = 0.0
    best_epoch = -1

    best_path = os.path.join(
        CKPT_DIR,
        f"stage1_{val_fold}_best_{save_best_by}.pth"
    )

    print("\nLoss weights:")
    print(f"  alpha_cam = {alpha_cam}")
    print(f"  beta_fqus = {beta_fqus}")
    print(f"  beta_pqus = {beta_pqus}")
    print(f"  save_best_by = {save_best_by}")

    for epoch in range(1, EPOCHS + 1):
        t0 = time.time()

        tr_losses, tr_acc = train_one_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            ce_loss_fn=ce_loss_fn,
            cam_loss_fn=cam_loss_fn,
            fqus_loss_fn=fqus_loss_fn,
            pqus_loss_fn=pqus_loss_fn,
            device=device,
            alpha_cam=alpha_cam,
            beta_fqus=beta_fqus,
            beta_pqus=beta_pqus,
            grad_clip=grad_clip
        )

        va_losses, va_acc, y_true, y_pred = eval_one_epoch(
            model=model,
            loader=val_loader,
            ce_loss_fn=ce_loss_fn,
            cam_loss_fn=cam_loss_fn,
            fqus_loss_fn=fqus_loss_fn,
            pqus_loss_fn=pqus_loss_fn,
            device=device,
            alpha_cam=alpha_cam,
            beta_fqus=beta_fqus,
            beta_pqus=beta_pqus
        )

        sens, spec, f1, balanced_acc, cm = compute_final_metrics(y_true, y_pred)

        current_score = get_checkpoint_score(
            save_best_by=save_best_by,
            va_acc=va_acc,
            sens=sens,
            spec=spec,
            f1=f1,
            balanced_acc=balanced_acc
        )

        dt = time.time() - t0

        print(
            f"Epoch {epoch:02d}/{EPOCHS} | "
            f"Train Total {tr_losses['total']:.4f} "
            f"CCE {tr_losses['cls']:.4f} "
            f"CAM {tr_losses['cam']:.4f} "
            f"FQUS {tr_losses['fqus']:.4f} "
            f"PQUS {tr_losses['pqus']:.4f} "
            f"Acc {tr_acc*100:5.2f}% | "
            f"Val Total {va_losses['total']:.4f} "
            f"CCE {va_losses['cls']:.4f} "
            f"Acc {va_acc*100:5.2f}% "
            f"Sens {sens*100:5.2f}% "
            f"Spec {spec*100:5.2f}% "
            f"F1 {f1:.4f} "
            f"BAcc {balanced_acc*100:5.2f}% | "
            f"Score({save_best_by}) {current_score:.4f} | "
            f"{dt:.1f}s"
        )

        if current_score > best_score:
            best_score = current_score
            best_acc = va_acc
            best_epoch = epoch

            torch.save(
                {
                    "model_state": model.state_dict(),
                    "val_fold": val_fold,
                    "best_acc": float(best_acc),
                    "best_score": float(best_score),
                    "best_epoch": int(best_epoch),
                    "save_best_by": save_best_by,
                    "sens": float(sens),
                    "spec": float(spec),
                    "f1": float(f1),
                    "balanced_acc": float(balanced_acc),
                    "confusion_matrix": cm.tolist(),
                    "loss_weights": {
                        "alpha_cam": alpha_cam,
                        "beta_fqus": beta_fqus,
                        "beta_pqus": beta_pqus
                    },
                    "ce_weight_malignant": ce_weight_malignant,
                    "qus_feature_names": QUS_FEATURE_NAMES,
                    "qus_norm_feature_names": QUS_NORM_FEATURE_NAMES
                },
                best_path
            )

            print(
                f"  [Saved best checkpoint] {best_path} | "
                f"Epoch {best_epoch} | "
                f"Score({save_best_by})={best_score:.4f} | "
                f"Acc={best_acc*100:.2f}% "
                f"Sens={sens*100:.2f}% "
                f"Spec={spec*100:.2f}% "
                f"F1={f1:.4f} "
                f"BAcc={balanced_acc*100:.2f}%"
            )

    # ------------------------------------------------------------
    # 5. Evaluate best checkpoint
    # ------------------------------------------------------------
    state = torch.load(best_path, map_location=device, weights_only=False)
    model.load_state_dict(state["model_state"])

    va_losses, va_acc, y_true, y_pred = eval_one_epoch(
        model=model,
        loader=val_loader,
        ce_loss_fn=ce_loss_fn,
        cam_loss_fn=cam_loss_fn,
        fqus_loss_fn=fqus_loss_fn,
        pqus_loss_fn=pqus_loss_fn,
        device=device,
        alpha_cam=alpha_cam,
        beta_fqus=beta_fqus,
        beta_pqus=beta_pqus
    )

    sens, spec, f1, balanced_acc, cm = compute_final_metrics(y_true, y_pred)

    print("\n" + "=" * 80)
    print(f"[{val_fold}] Best Validation Results")
    print("=" * 80)
    print(f"Saved by      : {save_best_by}")
    print(f"Best epoch    : {state.get('best_epoch', best_epoch)}")
    print(f"Best score    : {state.get('best_score', best_score):.4f}")
    print(f"Val Acc       : {va_acc*100:.2f}%")
    print(f"Sensitivity   : {sens*100:.2f}%")
    print(f"Specificity   : {spec*100:.2f}%")
    print(f"Balanced Acc  : {balanced_acc*100:.2f}%")
    print(f"F1 Score      : {f1:.4f}")
    print("Confusion matrix [labels 0=Benign, 1=Malignant]:")
    print(cm)

    return {
        "fold": val_fold,
        "best_acc": va_acc,
        "sens": sens,
        "spec": spec,
        "balanced_acc": balanced_acc,
        "f1": f1,
        "best_score": state.get("best_score", best_score),
        "best_epoch": state.get("best_epoch", best_epoch),
        "save_best_by": save_best_by,
        "ckpt": best_path,
        "scaler_path": scaler_path,
        "svm_path": svm_path,
        "ce_weight_malignant": ce_weight_malignant,
        "loss_weights": {
            "alpha_cam": alpha_cam,
            "beta_fqus": beta_fqus,
            "beta_pqus": beta_pqus
        }
    }

In [ ]:
'''# ===============================
# Cell 15: Run Fold1 Only
# ===============================

# Run only Fold1
# Fold1 is the validation set; Fold2-Fold5 are training.
# This is a quick test to verify the training loop works and can overfit a fold.
# Later, we run all folds in a loop and aggregate results.

res_fold1 = run_fold(
    val_fold="Fold1",
    batch_size=16,      
    img_size=299,
    lr=1e-4,
    weight_decay=1e-4,
    num_workers=2,

    # Fixed auxiliary loss weights
    # Keep low so CCE remains dominant at first.
    alpha_cam=0.003,
    beta_fqus=0.003,
    beta_pqus=0.001,

    # Optional gradient clipping; can keep None first.
    grad_clip=None
)

print(res_fold1)

'''

'# ===============================\n# Cell 15: Run Fold1 Only\n# ===============================\n\n# Run only Fold1\n# Fold1 is the validation set; Fold2-Fold5 are training.\n\nres_fold1 = run_fold(\n    val_fold="Fold1",\n    batch_size=16,      \n    img_size=299,\n    lr=1e-4,\n    weight_decay=1e-4,\n    num_workers=2,\n\n    # Fixed auxiliary loss weights\n    # Keep low so CCE remains dominant at first.\n    alpha_cam=0.003,\n    beta_fqus=0.003,\n    beta_pqus=0.001,\n\n    # Optional gradient clipping; can keep None first.\n    grad_clip=None\n)\n\nprint(res_fold1)\n\n'

In [ ]:
# ===============================
# Cell 16: Run 5-fold CV
# ===============================

import torch
import pandas as pd
import numpy as np
import gc


folds = [f"Fold{i}" for i in range(1, 6)]
all_res = []

for f in folds:
    print("\n" + "#" * 90)
    print(f"Running validation fold: {f}")
    print("#" * 90)

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    res = run_fold(
        val_fold=f,
        batch_size=16,      
        img_size=299,
        lr=2e-4,
        weight_decay=1e-4,
        num_workers=2,


        alpha_cam=0.3,
        beta_fqus=0.03,
        beta_pqus=0.01,
        ce_weight_malignant=None,
        save_best_by="balanced_acc", # can also try "acc", "f1", "sens" but balanced_acc is a good overall choice for imbalanced data

        # grad_clip can be set to a value like 1.0 if you encounter unstable training, but for this dataset we did not need it,, try it in ATL.        
        # Through we introduce class-weighted cross-entropy loss, but we did not need it here,, but in ATL try it.
        grad_clip=None
    )

    all_res.append(res)

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


df = pd.DataFrame(all_res)

# Pretty percentage columns
df["acc_%"] = df["best_acc"] * 100.0
df["sens_%"] = df["sens"] * 100.0
df["spec_%"] = df["spec"] * 100.0

print("\nPer-fold results:")
print(df[["fold", "acc_%", "sens_%", "spec_%", "f1"]].round(4))


def mean_std(series):
    return f"{series.mean():.2f} ± {series.std():.2f}"


print("\n5-Fold Summary (mean ± std):")
print("Accuracy (%)   :", mean_std(df["acc_%"]))
print("Sensitivity (%):", mean_std(df["sens_%"]))
print("Specificity (%):", mean_std(df["spec_%"]))
print("F1             :", mean_std(df["f1"]))

# Save to CSV for later
out_csv = "/kaggle/working/stage1_5fold_results_run1.csv"
df.to_csv(out_csv, index=False)

print(f"\nSaved per-fold results to: {out_csv}")

df



##########################################################################################
Running validation fold: Fold1
##########################################################################################

Starting fold: Fold1
Fold-wise QUS preparation for validation fold: Fold1
Train samples: 157
Val samples: 40

SVM classes: [0 1]

Train svm_qus_prob summary:
count    157.000000
mean       0.293096
std        0.369684
min        0.002617
25%        0.013949
50%        0.039268
75%        0.743624
max        0.973773
Name: svm_qus_prob, dtype: float64

Val svm_qus_prob summary:
count    40.000000
mean      0.277887
std       0.349642
min       0.004708
25%       0.014209
50%       0.069356
75%       0.573014
max       0.971994
Name: svm_qus_prob, dtype: float64

Normalized QUS columns:
['ED_N', 'SAF_N', 'TH_N', 'ASR_N', 'CVX_N', 'SOLIDITY_N', 'CCONTRAST_N', 'LBD_N']
[Saved] QUS scaler: /kaggle/working/qus_scaler_Fold1.joblib
[Saved] QUS SVM   : /kaggle/working/qus_svm_Fold1.j

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth
100%|██████████| 44.7M/44.7M [00:00<00:00, 184MB/s]
Downloading: "https://download.pytorch.org/models/inception_v3_google-0cc3c7bd.pth" to /root/.cache/torch/hub/checkpoints/inception_v3_google-0cc3c7bd.pth
100%|██████████| 104M/104M [00:00<00:00, 168MB/s] 
Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth
100%|██████████| 97.8M/97.8M [00:00<00:00, 195MB/s]
Downloading: "https://download.pytorch.org/models/vgg16_bn-6c64b313.pth" to /root/.cache/torch/hub/checkpoints/vgg16_bn-6c64b313.pth
100%|██████████| 528M/528M [00:02<00:00, 226MB/s]


[CE] Using normal CrossEntropyLoss

Loss weights:
  alpha_cam = 0.3
  beta_fqus = 0.03
  beta_pqus = 0.01
  save_best_by = balanced_acc
Epoch 01/50 | Train Total 0.7829 CCE 0.7052 CAM 0.1367 FQUS 0.9943 PQUS 0.6834 Acc 61.11% | Val Total 0.7593 CCE 0.6947 Acc 27.50% Sens 100.00% Spec  0.00% F1 0.4314 BAcc 50.00% | Score(balanced_acc) 0.5000 | 27.0s
  [Saved best checkpoint] /kaggle/working/stage1_Fold1_best_balanced_acc.pth | Epoch 1 | Score(balanced_acc)=0.5000 | Acc=27.50% Sens=100.00% Spec=0.00% F1=0.4314 BAcc=50.00%
Epoch 02/50 | Train Total 0.7275 CCE 0.6842 CAM 0.0213 FQUS 1.0089 PQUS 0.6661 Acc 59.03% | Val Total 0.7768 CCE 0.7188 Acc 27.50% Sens 100.00% Spec  0.00% F1 0.4314 BAcc 50.00% | Score(balanced_acc) 0.5000 | 13.7s
Epoch 03/50 | Train Total 0.5654 CCE 0.5388 CAM -0.0361 FQUS 1.0223 PQUS 0.6790 Acc 72.22% | Val Total 0.7575 CCE 0.7171 Acc 30.00% Sens 100.00% Spec  3.45% F1 0.4400 BAcc 51.72% | Score(balanced_acc) 0.5172 | 13.7s
  [Saved best checkpoint] /kaggle/working/s

,fold,best_acc,sens,spec,balanced_acc,f1,best_score,best_epoch,save_best_by,ckpt,scaler_path,svm_path,ce_weight_malignant,loss_weights,acc_%,sens_%,spec_%
0,Fold1,0.925000,0.727273,1.000000,0.863636,0.842105,0.863636,34,balanced_acc,/kaggle/working/stage1_Fold1_best_balanced_acc...,/kaggle/working/qus_scaler_Fold1.joblib,/kaggle/working/qus_svm_Fold1.joblib,None,"{'alpha_cam': 0.3, 'beta_fqus': 0.03, 'beta_pq...",92.500000,72.727273,100.000000
1,Fold2,0.975000,1.000000,0.965517,0.982759,0.956522,0.982759,15,balanced_acc,/kaggle/working/stage1_Fold2_best_balanced_acc...,/kaggle/working/qus_scaler_Fold2.joblib,/kaggle/working/qus_svm_Fold2.joblib,None,"{'alpha_cam': 0.3, 'beta_fqus': 0.03, 'beta_pq...",97.500000,100.000000,96.551724
2,Fold3,0.948718,0.909091,0.964286,0.936688,0.909091,0.936688,9,balanced_acc,/kaggle/working/stage1_Fold3_best_balanced_acc...,/kaggle/working/qus_scaler_Fold3.joblib,/kaggle/working/qus_svm_Fold3.joblib,None,"{'alpha_cam': 0.3, 'beta_fqus': 0.03, 'beta_pq...",94.871795,90.909091,96.428571
3,Fold4,0.871795,0.727273,0.928571,0.827922,0.761905,0.827922,43,balanced_acc,/kaggle/working/stage1_Fold4_best_balanced_acc...,/kaggle/working/qus_scaler_Fold4.joblib,/kaggle/working/qus_svm_Fold4.joblib,None,"{'alpha_cam': 0.3, 'beta_fqus': 0.03, 'beta_pq...",87.179487,72.727273,92.857143
4,Fold5,0.948718,1.000000,0.928571,0.964286,0.916667,0.964286,41,balanced_acc,/kaggle/working/stage1_Fold5_best_balanced_acc...,/kaggle/working/qus_scaler_Fold5.joblib,/kaggle/working/qus_svm_Fold5.joblib,None,"{'alpha_cam': 0.3, 'beta_fqus': 0.03, 'beta_pq...",94.871795,100.000000,92.857143
